
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" style="width: 600px">
</div>

# 2.1 Lecture - Building SQL ETL Pipelines

## Overview

Legacy ingestion and transformation patterns: full directory scans, static views, and scheduled CTAS rebuilds. They waste compute and deliver stale data. This lecture introduces the two declarative building blocks that replace them: <strong style="font-size: 14pt;">Streaming Tables</strong> and <strong style="font-size: 14pt;">Materialized Views</strong>.

Legacy patterns include ingestion scripts that re-list entire directories on every run, static views that recompute terabytes on every dashboard refresh, and CTAS jobs that fully rebuild Gold tables even when only a handful of rows changed. Streaming Tables and Materialized Views solve these problems:

<ul style="font-size: 14pt; line-height: 1.6; color: #333; margin: 8px 0 12px 0; padding-left: 24px;">
<li style="font-size: 14pt; margin-bottom: 4px;"><strong style="font-size: 14pt;">Streaming Tables</strong> ingest incrementally via Auto Loader. Only new files are processed on each refresh</li>
<li style="font-size: 14pt; margin-bottom: 4px;"><strong style="font-size: 14pt;">Materialized Views</strong> pre-compute and incrementally refresh transformations via the Enzyme engine. Only changed rows are reprocessed</li>
</ul>

<p style="font-size: 14pt; line-height: 1.6; color: #333;">Both are serverless, declarative, and SQL-native.</p>

<p style="font-size: 14pt; line-height: 1.6; color: #333;">This lecture covers five sections that build on each other:</p>

<p style="font-size: 14pt;"><strong style="font-size: 14pt;">A. Table Types</strong>: the four table types in the data warehouse</p>
<p style="font-size: 14pt;"><strong style="font-size: 14pt;">B. Streaming Tables</strong>: incremental ingestion via Auto Loader, with quality expectations</p>
<p style="font-size: 14pt;"><strong style="font-size: 14pt;">C. Materialized Views</strong>: pre-computed, incrementally refreshed transformations</p>
<p style="font-size: 14pt;"><strong style="font-size: 14pt;">D. Choosing the Right Tool</strong>: ST vs MV decision framework</p>
<p style="font-size: 14pt;"><strong style="font-size: 14pt;">E. Operationalizing</strong>: EXPLAIN, REFRESH POLICY, and SCHEDULE</p>


### Learning Objectives

<p style="font-size: 14pt; line-height: 1.6; color: #333;">By the end of this lecture, you will be able to:</p>
<ul style="font-size: 14pt; line-height: 1.6; color: #333; margin: 8px 0 12px 0; padding-left: 24px;">
<li style="font-size: 14pt; margin-bottom: 4px;">Create a Streaming Table that incrementally ingests data from cloud storage using Auto Loader</li>
<li style="font-size: 14pt; margin-bottom: 4px;">Explain why Auto Loader replaces COPY INTO for file ingestion: exactly-once guarantees, schema evolution, and scalable file discovery</li>
<li style="font-size: 14pt; margin-bottom: 4px;">Create a Materialized View that pre-computes and incrementally refreshes Silver-to-Gold transformations</li>
<li style="font-size: 14pt; margin-bottom: 4px;">Explain why Materialized Views replace static Views: pre-computed results, Enzyme incremental refresh, and cost reduction</li>
<li style="font-size: 14pt; margin-bottom: 4px;">Use EXPLAIN CREATE MATERIALIZED VIEW to predict whether an MV will incrementalize before deploying</li>
<li style="font-size: 14pt; margin-bottom: 4px;">Choose the right REFRESH POLICY (AUTO, INCREMENTAL, INCREMENTAL STRICT, FULL) based on EXPLAIN results and cost requirements</li>
<li style="font-size: 14pt; margin-bottom: 4px;">Configure SCHEDULE EVERY, SCHEDULE CRON, or TRIGGER ON UPDATE to automate MV refresh timing</li>
<li style="font-size: 14pt; margin-bottom: 4px;">Choose between Streaming Table and Materialized View for a given use case</li>
<li style="font-size: 14pt; margin-bottom: 4px;">Describe how Materialized Views and Streaming Tables are powered</li>
</ul>

## A. Table Types in the Data Warehouse

<p style="font-size: 14pt; line-height: 1.6; color: #333;">Before we look at Streaming Tables and Materialized Views, here is a quick overview of all four table types in the data warehouse.</p>

### A1. Four Table Types in the Data Warehouse

<p style="font-size: 14pt; line-height: 1.6; color: #333;">The data warehouse gives you four distinct table types, each designed for a different purpose. This lecture focuses on the last two. <strong style="font-size: 14pt;">Streaming Tables</strong> and <strong style="font-size: 14pt;">Materialized Views</strong>: but it helps to see where they fit alongside the types you already know.</p>

<div style="max-width: 100%; margin: 0 auto;">
<table style="width: 100%; border-collapse: collapse; line-height: 1.6;">
  <tr style="background: #272822; color: #fff;">
    <th style="padding: 12px 16px; text-align: left; border: 1px solid #2a4a54; font-size: 20px !important; width: 20%;">Type</th>
    <th style="padding: 12px 16px; text-align: left; border: 1px solid #2a4a54; font-size: 20px !important; width: 25%;">Created With</th>
    <th style="padding: 12px 16px; text-align: left; border: 1px solid #2a4a54; font-size: 20px !important; width: 15%;">Data Stored?</th>
    <th style="padding: 12px 16px; text-align: left; border: 1px solid #2a4a54; font-size: 20px !important; width: 40%;">Best For</th>
  </tr>
  <tr style="background: #F9F7F4;">
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;"><strong style="font-size: 14pt;">Managed Table</strong><br/><small style="color:#666;">(Delta or Iceberg)</small></td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;"><code style="font-size: 14pt;">CREATE TABLE</code></td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;">Yes</td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;">General-purpose storage with full CRUD: inserts, updates, deletes, merges</td>
  </tr>
  <tr style="background: #fff;">
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;"><strong style="font-size: 14pt;">View</strong></td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;"><code style="font-size: 14pt;">CREATE VIEW</code></td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;">No</td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;">Logical aliases: query re-executes every time, no storage cost but no performance gain</td>
  </tr>
  <tr style="background: #f0f7ff;">
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;"><strong style="font-size: 14pt;">Streaming Table</strong></td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;"><code style="font-size: 14pt;">CREATE STREAMING TABLE</code></td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;">Yes</td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;">Incremental ingestion of <strong>append-only</strong> data from files via Auto Loader. It processes only new files on each refresh. For non-append patterns (inserts, updates, deletes), use AUTO CDC with a Streaming Table</td>
  </tr>
  <tr style="background: #f7fdf4;">
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;"><strong style="font-size: 14pt;">Materialized View</strong></td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;"><code style="font-size: 14pt;">CREATE MATERIALIZED VIEW</code></td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;">Yes</td>
    <td style="padding: 12px 16px; border: 1px solid #E8E3DC; font-size: 20px !important;">Pre-computed query results that <strong>incrementally refresh</strong>: only reprocesses changed rows, not the full dataset</td>
  </tr>
</table>
</div>

<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">The Two You Already Know</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Managed Tables</strong> (Delta or Iceberg) are the workhorses: you <code style="font-size: 14pt;">CREATE TABLE</code>, insert data, update rows, delete records, and run merges. They store data physically and support full ACID transactions. You used these in Demo 1.2 when you explored the course dataset.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Views</strong> are just saved queries. <code style="font-size: 14pt;">CREATE VIEW</code> stores the SQL definition, not the results. Every time you query a view, the underlying SQL re-executes against the source tables. Zero storage cost, but also zero performance benefit. A view over a terabyte table scans a terabyte every time.</li>
        </ul>
        <strong style="color: #1B5162;">The Two This Lecture Covers</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Streaming Tables</strong> solve the ingestion problem. Instead of re-listing an entire directory of files on every run (like <code style="font-size: 14pt;">COPY INTO</code>), a Streaming Table uses <strong style="font-size: 14pt;">Auto Loader</strong> to track which files have already been processed and only pick up new ones. You write one <code style="font-size: 14pt;">CREATE STREAMING TABLE</code> statement and the platform handles incremental file discovery, schema inference, and exactly-once processing.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Materialized Views</strong> solve the transformation problem. Instead of re-running an expensive join or aggregation every time a dashboard refreshes (like a View does), an MV stores the pre-computed result and uses the <strong style="font-size: 14pt;">Enzyme engine</strong> to incrementally refresh only the rows that changed upstream. The result is faster queries and lower compute cost.</li>
        </ul>
        <strong style="color: #1B5162;">Why This Matters for the Course</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">The ETL pipeline you are building in this course uses <strong style="font-size: 14pt;">Streaming Tables</strong> for the Bronze layer (ingestion from files) and <strong style="font-size: 14pt;">Materialized Views</strong> for Silver and Gold layers (transformations and aggregations). Managed Tables hold the raw source data, and Views may appear for ad-hoc queries, but the pipeline backbone is ST + MV.</li>
          <li style="font-size: 14pt;">Both Streaming Tables and Materialized Views are governed by <strong style="font-size: 14pt;">Unity Catalog</strong> you saw the auto-created pipeline in Demo 1.2 and Lab 1.3.</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>

## B. Streaming Tables: Incremental Ingestion

<p style="font-size: 14pt; line-height: 1.6; color: #333;">Streaming Tables are the modern, declarative way to ingest data from cloud storage into Databricks. This section introduces what a Streaming Table is and why it replaces the older COPY INTO command.</p>

### B1. What Is a Streaming Table?

<p style="font-size: 14pt; line-height: 1.6; color: #333;">A <strong style="font-size: 14pt;">Streaming Table</strong> is a Unity Catalog managed table with built-in support for <strong>incremental data processing</strong>: it knows how to pick up new files from cloud storage, processing each one exactly once.</p>

<!-- ── Visual: button-expand-panels ── -->

#### Select each feature to explore how Streaming Tables work

<br/>

<div style="max-width: 100%; margin: 0 auto; font-family: sans-serif">

<style>
.st1-row { display: block; white-space: nowrap; font-size: 0; width: 100%; box-sizing: border-box; }
.st1-box { display: inline-block; width: 23.5%; margin-right: 2%; min-height: 80px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 12px 8px; text-align: center; box-sizing: border-box; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; }
.st1-box:last-child { margin-right: 0; }
.st1-box:hover { transform: translateY(-2px); }
.st1-box.st1-active { background: #fff; border-left-color: var(--st1-c); border-right-color: var(--st1-c); border-bottom-color: var(--st1-c); }
.st1-label { display: block; font-size: 14pt; font-weight: 700; color: #0b2026; line-height: 1.3; white-space: normal; pointer-events: none; }
.st1-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; }
.st1-detail-wrap.st1-open { max-height: 1200px; opacity: 1; margin-top: 12px; }
.st1-detail-card { background: #F9F7F4; border-radius: 10px; padding: 20px; box-sizing: border-box; border-top: 7px solid #ccc; }
.st1-can-block { border-left: 4px solid; border-radius: 0 6px 6px 0; padding: 14px 14px; margin-bottom: 14px; font-size: 14pt; line-height: 1.6; }
.st1-can-block ul { margin: 8px 0 0 0; padding-left: 20px; }
.st1-can-block li { margin-bottom: 8px; }
.st1-example-block { border-left: 4px solid; border-radius: 0 6px 6px 0; padding: 12px 14px; font-size: 14pt; line-height: 1.5; }
.st1-example-block pre { background: #272822; color: #f8f8f2; border-radius: 6px; padding: 12px 14px; font-family: monospace; font-size: 14pt; line-height: 1.6; margin: 8px 0 0 0; white-space: pre; overflow-x: auto; }
.st1-example-block .kw { color: #66d9ef; font-weight: 600; }
.st1-example-block .fn { color: #a6e22e; }
.st1-example-block .str { color: #e6db74; }
</style>

<div style="background: #F8F9FC; border: 3px solid #1B5162; border-radius: 10px; padding: 24px; box-sizing: border-box">

  <div class="st1-row">
    <span class="st1-box st1-active" data-st1-id="0" onclick="st1Sel(0)" style="--st1-c:#2574B5; border-top-color:#2574B5">
      <span class="st1-label"> The SQL</span>
    </span><span class="st1-box" data-st1-id="1" onclick="st1Sel(1)" style="--st1-c:#1B5162; border-top-color:#1B5162">
      <span class="st1-label"> Auto Loader</span>
    </span><span class="st1-box" data-st1-id="2" onclick="st1Sel(2)" style="--st1-c:#FF5F46; border-top-color:#FF5F46">
      <span class="st1-label"> Data Reliability</span>
    </span><span class="st1-box" data-st1-id="3" onclick="st1Sel(3)" style="--st1-c:#FFAB00; border-top-color:#FFAB00">
      <span class="st1-label"> Serverless</span>
    </span>
  </div>

  <div class="st1-detail-wrap" id="st1-detail-wrap">
    <div class="st1-detail-card" id="st1-detail-card"></div>
  </div>

</div>

</div>

<script>
var ST1_ITEMS = [
  { title: "The SQL You Write", color: "#2574B5", bg: "rgba(37,116,181,0.10)",
    content: `<strong style="font-size: 14pt;">One SQL statement</strong> creates a governed, incremental ingestion pipeline:<ul><li style="font-size: 14pt;"><code style="font-size: 14pt;">STREAMING TABLE</code>: a <strong>UC managed table</strong> with streaming semantics</li><li style="font-size: 14pt;"><code style="font-size: 14pt;">STREAM read_files()</code>: <strong>Auto Loader in SQL</strong>, discovering new files automatically</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Schema inferred</strong> from source. <strong>no column definitions needed</strong> for JSON, Parquet, CSV, Avro</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Schema evolution</strong>: new columns added <strong>automatically</strong>; mismatches land in <code style="font-size: 14pt;">_rescued_data</code></li></ul>`,
    example: `<strong style="font-size: 14pt;">Create a Streaming Table:</strong><pre style="background:#272822;color:#f8f8f2;border-radius:6px;padding:12px 14px;font-family:monospace; line-height:1.6;margin:8px 0 0 0;white-space:pre;overflow-x:auto"><span style="font-size: 14pt; color:#66d9ef;font-weight:600">CREATE OR REFRESH STREAMING TABLE</span> bronze_orders\n<span style="font-size: 14pt; color:#66d9ef;font-weight:600">AS SELECT</span> *\n<span style="font-size: 14pt; color:#66d9ef;font-weight:600">FROM STREAM</span> <span style="font-size: 14pt; color:#a6e22e">read_files</span>(\n  '/Volumes/catalog/schema/raw/orders/',\n  format =&gt; 'json',\n  header =&gt; 'true'\n);</pre>` },
  { title: "Auto Loader \u2014 File Discovery", color: "#1B5162", bg: "rgba(2,163,111,0.10)",
    content: `<code style="font-size: 14pt;">read_files()</code> is the SQL interface to <strong style="font-size: 14pt;">Auto Loader</strong>: the same engine as Python&#39;s <code style="font-size: 14pt;">cloudFiles</code>, with a friendlier SQL front door.<ul><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Directory listing</strong> (default) or <strong style="font-size: 14pt;">cloud-native file notification</strong> for discovering <strong>new files</strong></li><li style="font-size: 14pt;">Each <code style="font-size: 14pt;">REFRESH</code> processes <strong>only files that arrived since the last run</strong></li><li style="font-size: 14pt;">Scales to <strong style="font-size: 14pt;">millions of files</strong> with directory listing; for <strong style="font-size: 14pt;">billions of files</strong>, enable file notification mode</li><li style="font-size: 14pt;">&#x25C6; At <a href="https://www.databricks.com/customers/sciplay" style="color:#1B5162">SciPlay</a>, Auto Loader delivered <strong style="font-size: 14pt;"><strong>60x faster</strong> time-to-insight</strong> (4 hours &#8594; 4 minutes)</li></ul>`,
    example: `<strong style="font-size: 14pt;">Auto Loader with options:</strong><pre style="background:#272822;color:#f8f8f2;border-radius:6px;padding:12px 14px;font-family:monospace; line-height:1.6;margin:8px 0 0 0;white-space:pre;overflow-x:auto"><span style="font-size: 14pt; color:#66d9ef;font-weight:600">CREATE OR REFRESH STREAMING TABLE</span> bronze_events\n<span style="font-size: 14pt; color:#66d9ef;font-weight:600">AS SELECT</span> *\n<span style="font-size: 14pt; color:#66d9ef;font-weight:600">FROM STREAM</span> <span style="font-size: 14pt; color:#a6e22e">read_files</span>(\n  '/Volumes/catalog/schema/raw/events/',\n  format =&gt; 'json',\n  header =&gt; 'true',\n  <span style="font-size: 14pt; color:#a6e22e">schemaEvolutionMode</span> =&gt; 'addNewColumns',\n  <span style="font-size: 14pt; color:#a6e22e">inferColumnTypes</span> =&gt; true\n);</pre>` },
  { title: "Data Reliability", color: "#FF5F46", bg: "rgba(255,95,70,0.10)",
    content: `Streaming Tables provide four layers of data reliability out of the box:<ul><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Exactly-once processing</strong>: a persistent RocksDB checkpoint stamps every file as &#8220;received&#8221; so it is <strong>never processed twice</strong>. Survives restarts, failures, and reschedules</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Schema evolution</strong>: new source columns are added to the table <strong>automatically</strong> as they appear. No manual <code style="font-size: 14pt;">ALTER TABLE</code> required</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Rescued data column</strong>: <code style="font-size: 14pt;">_rescued_data</code> captures any fields that don&#39;t match the known schema as JSON, so <strong>nothing is silently dropped</strong>. It&#39;s your &#8220;lost and found&#8221; bin</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Unity Catalog governed</strong>: every Streaming Table is a UC managed table with <strong>permissions, tags, lineage, and audit logging</strong> built in</li></ul>`,
    example: `<strong style="font-size: 14pt;">Bottom line:</strong> Built-in reliability with zero defensive code.` },
  { title: "Serverless SDP Pipeline", color: "#FFAB00", bg: "rgba(255,171,0,0.10)",
    content: `Every <code style="font-size: 14pt;">CREATE</code> or <code style="font-size: 14pt;">REFRESH</code> is powered by a <strong style="font-size: 14pt;">serverless SDP (Spark Declarative Pipelines)</strong> pipeline. The SQL warehouse dispatches the command, and the serverless pipeline does all the heavy lifting independently.<ul><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Separate compute</strong>: refresh processing runs on the <strong>pipeline</strong>, not on your <strong>warehouse</strong></li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Not continuous</strong>: processes on <strong>REFRESH only</strong> (manual, scheduled, or triggered)</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Automatic scaling</strong>: the serverless pipeline <strong>right-sizes compute</strong> for each refresh, no cluster configuration needed</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Zero infrastructure</strong>: <strong>no clusters</strong> to configure, start, or monitor. The platform provisions and tears down compute <strong>automatically</strong></li></ul>`,
    example: `&#x25C6; <strong style="font-size: 14pt;">At <a href="https://www.databricks.com/customers/block/lakeflow-declarative-pipelines" style="color:#FFAB00">Block</a></strong>, switching to serverless streaming tables reduced pipeline development from <strong style="font-size: 14pt;">days to hours</strong> and improved autoscaling efficiency.` }
];
var st1Current = null;
function st1Sel(id) {
  var wrap = document.getElementById("st1-detail-wrap");
  var card = document.getElementById("st1-detail-card");
  var c = ST1_ITEMS[id];
  document.querySelectorAll(".st1-box").forEach(function(b) { b.classList.toggle("st1-active", parseInt(b.dataset.st1Id,10)===id); });
  if (st1Current === id) { wrap.classList.remove("st1-open"); document.querySelectorAll(".st1-box").forEach(function(b){b.classList.remove("st1-active");}); st1Current=null; return; }
  st1Current = id;
  card.style.borderTopColor = c.color;
  card.innerHTML = '<div style="font-size:18pt;font-weight:700;margin-bottom:12px;color:#0b2026;">'+c.title+'</div><div class="st1-can-block" style="background:'+c.bg+';border-color:'+c.color+'">'+c.content+'</div><div class="st1-example-block" style="background:'+c.bg+';border-color:'+c.color+'">'+c.example+'</div>';
  wrap.classList.add("st1-open");
}
st1Sel(0);
</script>
<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">The SQL You Write</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">A <strong style="font-size: 14pt;">Streaming Table</strong> is a Unity Catalog managed table. It appears in Catalog Explorer, respects permissions, and supports tags, lineage, and all governance features like any other table</li>
          <li style="font-size: 14pt;">The <code style="font-size: 14pt;">STREAM</code> keyword tells the engine to use streaming semantics: track which files have been processed and only pick up new ones on each refresh</li>
          <li style="font-size: 14pt;"><code style="font-size: 14pt;">read_files()</code> is the SQL interface to Auto Loader, the same engine as Python&#39;s <code style="font-size: 14pt;">cloudFiles</code> format, just with a friendlier SQL front door</li>
          <li style="font-size: 14pt;">You write one <code style="font-size: 14pt;">CREATE STREAMING TABLE</code> statement: no file listing logic, no checkpoint management, no schema definitions</li>
        </ul>
        <strong style="color: #1B5162;">Auto Loader: File Discovery</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Auto Loader discovers new files via <strong style="font-size: 14pt;">directory listing</strong> (default) or <strong style="font-size: 14pt;">cloud-native file notification</strong> for higher scale</li>
          <li style="font-size: 14pt;">Each <code style="font-size: 14pt;">REFRESH</code> processes only files that arrived since the last run; the RocksDB checkpoint tracks exactly where it left off</li>
          <li style="font-size: 14pt;">Scales to <strong style="font-size: 14pt;">millions of files</strong> with the default directory listing mode; for <strong style="font-size: 14pt;">billions of files</strong>, enable file notification mode for constant-time discovery. Unlike COPY INTO&#39;s file-tracking state which becomes expensive at scale</li>
          <li style="font-size: 14pt;">&#x25C6; At <a style="font-size: 14pt;" href="https://www.databricks.com/customers/sciplay">SciPlay</a>, Auto Loader delivered a <strong style="font-size: 14pt;">60x improvement in time-to-insight</strong>: from 4 hours to 4 minutes for gaming analytics pipelines</li>
        </ul>
        <strong style="color: #1B5162;">Data Reliability</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Exactly-once processing</strong>: a persistent RocksDB checkpoint stamps every file as "received" so it is never processed twice. Survives restarts, failures, and reschedules</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Schema inference</strong>: Auto Loader infers the schema from the first batch of files, no column definitions needed for JSON, Parquet, CSV, or Avro</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Schema evolution</strong>: new source columns are added to the table automatically as they appear. No manual <code style="font-size: 14pt;">ALTER TABLE</code> required</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Rescued data</strong>: the <code style="font-size: 14pt;">_rescued_data</code> column captures any fields that don&#39;t match the known schema as JSON, so nothing is silently dropped. It&#39;s your "lost and found" bin</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Unity Catalog governed</strong>: permissions, tags, lineage, and audit logging from the moment the table is created</li>
        </ul>
        <strong style="color: #1B5162;">Serverless SDP Pipeline</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Every <code style="font-size: 14pt;">CREATE</code> or <code style="font-size: 14pt;">REFRESH</code> is powered by a <strong style="font-size: 14pt;">serverless Spark Declarative Pipeline</strong>: the SQL warehouse dispatches the command; the pipeline does all the heavy lifting independently</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Separate compute</strong>: refresh processing runs on the pipeline, not on your warehouse</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Automatic scaling</strong>: the serverless pipeline right-sizes compute for each refresh, no cluster configuration needed</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Zero infrastructure</strong>: no clusters to configure, start, or monitor. The platform provisions and tears down compute automatically</li>
          <li style="font-size: 14pt;">&#x25C6; At <a style="font-size: 14pt;" href="https://www.databricks.com/customers/block/lakeflow-declarative-pipelines">Block</a>, switching to serverless streaming tables reduced pipeline development from <strong style="font-size: 14pt;">days to hours</strong> and improved autoscaling efficiency &#x25C6;</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>
<script>  
<!-- Retry any images that failed to render on initial load (500ms and 1500ms delays -->
[500, 1500].forEach(function(delay) {
    setTimeout(function() {
        document.querySelectorAll('.two-pane img').forEach(function(img) {
            if (!img.complete || img.naturalWidth === 0) {                                                                                                                                
                img.src = img.src;
            }                                                                                                                                                                             
        });     
    }, delay);
});
</script>
<script>  
<!-- Retry any images that failed to render on initial load (500ms and 1500ms delays -->
[500, 1500].forEach(function(delay) {
    setTimeout(function() {
        document.querySelectorAll('.two-pane img').forEach(function(img) {
            if (!img.complete || img.naturalWidth === 0) {                                                                                                                                
                img.src = img.src;
            }                                                                                                                                                                             
        });     
    }, delay);
});
</script>

### B2. Why Auto Loader Replaces COPY INTO

Section B1 introduced <code style="font-size: 14pt;">read_files()</code>: the SQL interface to <strong style="font-size: 14pt;">Auto Loader</strong>. Auto Loader is the ingestion engine that replaces COPY INTO, and Streaming Tables let you use it with a single SQL statement. COPY INTO is not deprecated, but Auto Loader addresses the key limitations that made COPY INTO difficult at scale.

<!-- ── Visual: comparison-matrix ── -->
<div class="a2cm-wrap">
<style>
.a2cm-wrap { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 100%; margin: 0 auto; }
.a2cm-title { color: #1B3139; font-size: 17pt; font-weight: 700; margin: 0 0 16px 0; text-align: center; }
.a2cm-table { width: 100%; border-collapse: collapse; border-radius: 8px; overflow: hidden; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border: 1px solid #E8E3DC; }
.a2cm-table thead th { background: #272822; color: white; font-size: 14pt; font-weight: 700; padding: 14px 18px; text-align: center; border-bottom: 2px solid #1B5162; border-right: 2px solid #3a5a64; }
.a2cm-table thead th:first-child { text-align: left; min-width: 200px; }
.a2cm-table tbody td { padding: 13px 18px; font-size: 14pt; text-align: center; border-bottom: 1px solid #E8E3DC; border-right: 1px solid #E8E3DC; color: #1B3139; }
.a2cm-table tbody td:first-child { text-align: left; font-weight: 600; }
.a2cm-table tbody tr:nth-child(odd) { background: #F9F7F4; }
.a2cm-table tbody tr:nth-child(even) { background: white; }
.a2cm-table tbody tr:hover { background: #EDE9E3; }
.a2cm-check { color: #00A972; font-weight: 700; font-size: 15pt; }
.a2cm-cross { color: #FF3621; font-weight: 700; font-size: 15pt; }
.a2cm-partial { color: #C68B00; font-weight: 700; font-size: 14pt; }
</style>
<h2 class="a2cm-title">COPY INTO vs. Auto Loader (Streaming Tables)</h2>
<table class="a2cm-table">
  <thead><tr>
    <th>Capability</th>
    <th>COPY INTO</th>
    <th>Streaming Table</th>
  </tr></thead>
  <tbody>
    <tr><td>File Discovery</td><td>Directory listing (slows as files grow)</td><td><span class="a2cm-check"></span> Directory listing (default) or file notification. Scales with either mode</td></tr>
    <tr><td>Scale at Millions of Files</td><td><span class="a2cm-cross"></span> Degrades</td><td><span class="a2cm-check"></span> Scales well; for billions of files, enable file notification mode</td></tr>
    <tr><td>Exactly-Once Guarantees</td><td><span class="a2cm-partial"></span> State tables</td><td><span class="a2cm-check"></span> RocksDB checkpoint</td></tr>
    <tr><td>Schema Inference</td><td><span class="a2cm-partial"></span> Limited (INFER_SCHEMA option)</td><td><span class="a2cm-check"></span> Automatic</td></tr>
    <tr><td>Schema Evolution</td><td><span class="a2cm-partial"></span> Limited mergeSchema</td><td><span class="a2cm-check"></span> Full + _rescued_data</td></tr>
    <tr><td>Compute Isolation</td><td>Runs on warehouse</td><td><span class="a2cm-check"></span> Serverless pipeline</td></tr>
    <tr><td>Watermarks Required?</td><td>N/A (not a streaming construct)</td><td><span class="a2cm-check"></span> Not for simple ingestion; required only for stateful operations (windowed aggregations, stream-stream joins)</td></tr>
  </tbody>
</table>
</div>

<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">File Discovery and Scale</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">COPY INTO</strong> scans the <strong>entire directory</strong> on every run and tracks loaded files via internal state tables. This becomes expensive as file counts grow into the tens of thousands</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Auto Loader</strong> defaults to <strong>directory listing mode</strong>, which scales well to millions of files. For billions of files, explicitly enable <strong>file notification mode</strong> (<code style="font-size: 14pt;">cloudFiles.useNotifications = true</code>) for constant-time file discovery</li>
          <li style="font-size: 14pt;">The difference is architectural: COPY INTO is like manually checking a mailbox and crossing off letters. Auto Loader is a digital mailroom that scans every delivery automatically and keeps a tamper-proof log</li>
        </ul>
        <strong style="color: #1B5162;">Exactly-Once and Compute Isolation</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Both provide <strong style="font-size: 14pt;">exactly-once processing</strong>: the difference is robustness. Auto Loader&#39;s RocksDB checkpoint is <strong>more resilient at scale</strong> than COPY INTO&#39;s file-tracking state</li>
          <li style="font-size: 14pt;">COPY INTO runs on the <strong>calling warehouse</strong>. Auto Loader runs on a <strong>serverless SDP pipeline</strong>: separate compute that doesn&#39;t compete with your queries</li>
          <li style="font-size: 14pt;">&#x25C6; At <a style="font-size: 14pt;" href="https://www.databricks.com/customers/block/lakeflow-declarative-pipelines">Block</a>, switching to Auto Loader with streaming tables reduced pipeline development from <strong style="font-size: 14pt;">days to hours</strong> &#x25C6;</li>
        </ul>
        <strong style="color: #1B5162;">Schema Inference, Evolution, and Rescued Data</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">COPY INTO offers limited <code style="font-size: 14pt;">mergeSchema</code> and an <code style="font-size: 14pt;">INFER_SCHEMA</code> option, but <strong>no automatic schema evolution</strong> and <strong>no rescue column</strong>: schema management remains largely manual</li>
          <li style="font-size: 14pt;">Auto Loader provides <strong style="font-size: 14pt;">full automatic inference</strong>, <strong style="font-size: 14pt;">automatic evolution</strong> (new columns added without <code style="font-size: 14pt;">ALTER TABLE</code>), and the <code style="font-size: 14pt;">_rescued_data</code> column for mismatched fields</li>
          <li style="font-size: 14pt;">A major real estate technology company experienced columns being silently dropped during COPY INTO ingestion, a data quality issue that <code style="font-size: 14pt;">_rescued_data</code> would have prevented</li>
        </ul>
        <strong style="color: #1B5162;">Watermarks and When to Still Use COPY INTO</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Common misconception: "Streaming Tables require watermarks." <strong style="font-size: 14pt;">False</strong> for file ingestion. Watermarks are only needed for stateful operations (windowed aggregations, stream-stream joins)</li>
          <li style="font-size: 14pt;">COPY INTO is <strong style="font-size: 14pt;">not deprecated</strong> and remains useful for: small-scale ad-hoc loads, re-processing specific file subsets, or environments without serverless compute</li>
          <li style="font-size: 14pt;">For <strong>production ingestion pipelines</strong>, Auto Loader via Streaming Tables is the recommended path forward</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>

### B3. Quality Expectations on Streaming Tables and Materialized Views


Raw data is messy: null IDs, negative prices, dates from the future. Without guardrails, bad records flow into Silver and Gold, causing broken dashboards and wrong business decisions. <strong style="font-size: 14pt;">Expectations</strong> let you declare quality rules directly in the <code style="font-size: 14pt;">CREATE STREAMING TABLE</code> statement (and, as of June 2026, in <code style="font-size: 14pt;">CREATE MATERIALIZED VIEW</code> as well), so validation happens inline as data flows through the pipeline, with no separate audit job needed. This section uses Streaming Tables to introduce the pattern; the same syntax applies to Materialized Views.

<!-- ── Visual: flip-cards + interactive flow diagram ── -->

#### Click each card to see the behavior and highlight the flow path

<br/>

<div class="dq-flip-wrap">
<style>
.dq-flip-wrap { max-width: 100%; margin: 0 auto; font-family: system-ui, -apple-system, sans-serif; }
.dq-flip-row { display: flex; gap: 16px; justify-content: center; perspective: 1200px; }
.dq-flip-card { flex: 1; min-width: 320px; height: 300px; perspective: 1000px; cursor: pointer; }
.dq-flip-inner { position: relative; width: 100%; height: 100%; transition: transform 0.6s cubic-bezier(0.4,0,0.2,1); transform-style: preserve-3d; }
.dq-flip-card.dq-flipped .dq-flip-inner { transform: rotateY(180deg); }
.dq-flip-front, .dq-flip-back { position: absolute; top: 0; left: 0; width: 100%; height: 100%; backface-visibility: hidden; -webkit-backface-visibility: hidden; border-radius: 10px; box-sizing: border-box; }
.dq-flip-front { background: #F9F7F4; border: 2px solid #E8E3DC; box-shadow: 0 2px 8px rgba(27,49,57,0.08); display: flex; flex-direction: column; align-items: center; justify-content: center; padding: 18px 16px; }
.dq-flip-front-title { font-size: 17pt; font-weight: 700; color: #fff; padding: 6px 18px; border-radius: 6px; margin-bottom: 12px; }
.dq-flip-front-code { background: #272822; color: #f5f5f5; border-radius: 6px; padding: 10px 14px; font-family: monospace; font-size: 16pt; line-height: 1.6; text-align: left; width: 100%; box-sizing: border-box; }
.dq-flip-front-hint { font-size: 16pt; color: #999; margin-top: 10px; }
.dq-flip-back { transform: rotateY(180deg); color: #fff; padding: 18px 18px; display: flex; flex-direction: column; justify-content: flex-start; border-radius: 10px; }
.dq-flip-back-title { font-size: 16pt; font-weight: 700; margin-bottom: 8px; }
.dq-flip-back-text { font-size: 16pt; line-height: 1.55; }
.dq-flip-back-text ul { margin: 4px 0 0; padding-left: 16px; }
.dq-flip-back-text li { margin-bottom: 4px; }
.dq-kw { color: #66d9ef; font-weight: 600; }
.dq-obj { color: #a6e22e; }
/* SVG highlight states */
.dq-svg-dim { opacity: 0.15; transition: opacity 0.4s; }
.dq-svg-bright { opacity: 1; transition: opacity 0.4s; }
</style>

<div class="dq-flip-row">
  <!-- Warn -->
  <div class="dq-flip-card" data-dq="warn" onclick="dqFlip(this,'warn')">
    <div class="dq-flip-inner">
      <div class="dq-flip-front">
        <div class="dq-flip-front-title" style="background:#2574B5;">Warn (Default)</div>
        <div class="dq-flip-front-code"><span class="dq-kw">CONSTRAINT</span> <span class="dq-obj">valid_amount</span><br/><span class="dq-kw">EXPECT</span> (amount &gt; 0)</div>
        <div class="dq-flip-front-hint">Click to flip</div>
      </div>
      <div class="dq-flip-back" style="background:#2574B5;">
        <div class="dq-flip-back-title">Warn: Behavior</div>
        <div class="dq-flip-back-text">Invalid records are <strong style="font-size: 14pt;">written to the target</strong>. Violations recorded in event_log.<ul><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Best for:</strong> Quality monitoring, non-critical fields</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Scenario:</strong> Track issues for the source team without blocking dashboards</li></ul></div>
      </div>
    </div>
  </div>
  <!-- Drop -->
  <div class="dq-flip-card" data-dq="drop" onclick="dqFlip(this,'drop')">
    <div class="dq-flip-inner">
      <div class="dq-flip-front">
        <div class="dq-flip-front-title" style="background:#E5A100;">Drop</div>
        <div class="dq-flip-front-code"><span class="dq-kw">CONSTRAINT</span> <span class="dq-obj">valid_amount</span><br/><span class="dq-kw">EXPECT</span> (amount &gt; 0)<br/><span class="dq-kw">ON VIOLATION DROP ROW</span></div>
        <div class="dq-flip-front-hint">Click to flip</div>
      </div>
      <div class="dq-flip-back" style="background:#C48600;">
        <div class="dq-flip-back-title">Drop: Behavior</div>
        <div class="dq-flip-back-text">Invalid records are <strong style="font-size: 14pt;">dropped before written</strong> to target.<ul><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Best for:</strong> Test records, known bad patterns, duplicates</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Scenario:</strong> Filter test orders with amount = 0 before the dashboard</li></ul></div>
      </div>
    </div>
  </div>
  <!-- Fail -->
  <div class="dq-flip-card" data-dq="fail" onclick="dqFlip(this,'fail')">
    <div class="dq-flip-inner">
      <div class="dq-flip-front">
        <div class="dq-flip-front-title" style="background:#D93025;">Fail</div>
        <div class="dq-flip-front-code"><span class="dq-kw">CONSTRAINT</span> <span class="dq-obj">valid_amount</span><br/><span class="dq-kw">EXPECT</span> (amount &gt; 0)<br/><span class="dq-kw">ON VIOLATION FAIL UPDATE</span></div>
        <div class="dq-flip-front-hint">Click to flip</div>
      </div>
      <div class="dq-flip-back" style="background:#B71C1C;">
        <div class="dq-flip-back-title">Fail: Behavior</div>
        <div class="dq-flip-back-text">Invalid records <strong style="font-size: 14pt;">prevent the update</strong>. Pipeline halts.<ul><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Best for:</strong> Financial data, regulatory reporting</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Scenario:</strong> Negative revenue stops everything. Wrong &gt; late</li></ul></div>
      </div>
    </div>
  </div>
</div>
</div>

<br/>

<!-- Pipeline Expectations Flow Graph with highlight paths -->
<div style="max-width: 100%; margin: 0 auto; font-family: system-ui, -apple-system, sans-serif;">
<div style="font-size: 15pt; font-weight: 700; color: #1B3139; text-align: center; margin-bottom: 12px;">Pipeline Expectations Flow Graph</div>
<svg id="dq-flow-svg" viewBox="0 0 920 310" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:920px;display:block;margin:0 auto;">
  <defs>
    <marker id="dqah" markerWidth="10" markerHeight="7" refX="9" refY="3.5" orient="auto"><polygon points="0 0, 10 3.5, 0 7" fill="#333"/></marker>
    <marker id="dqahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5" orient="auto"><polygon points="0 0, 10 3.5, 0 7" fill="#888"/></marker>
  </defs>
  <!-- Shared path: Raw Data -> Expectation -> Fail -> Action -->
  <g class="dq-path-shared">
    <rect x="10" y="105" width="100" height="40" rx="5" fill="#e8e8e8" stroke="#999" stroke-width="2"/>
    <text x="60" y="130" text-anchor="middle" font-size="13" font-weight="600" fill="#333">Raw Data</text>
    <line x1="110" y1="125" x2="168" y2="125" stroke="#333" stroke-width="2" marker-end="url(#dqah)"/>
    <polygon points="240,50 310,125 240,200 170,125" fill="#FFF3C4" stroke="#C8A415" stroke-width="2"/>
    <text x="240" y="130" text-anchor="middle" font-size="14" font-weight="700" fill="#333">Expectation</text>
    <line x1="275" y1="163" x2="380" y2="195" stroke="#333" stroke-width="2" marker-end="url(#dqah)"/>
    <text x="315" y="192" font-size="12" font-weight="600" fill="#555">Fail</text>
    <polygon points="440,155 490,200 440,245 390,200" fill="#FFF3C4" stroke="#C8A415" stroke-width="2"/>
    <text x="440" y="205" text-anchor="middle" font-size="12" font-weight="700" fill="#333">Action</text>
  </g>
  <!-- Pass path (always visible) -->
  <g class="dq-path-pass">
    <line x1="275" y1="88" x2="420" y2="55" stroke="#333" stroke-width="2" marker-end="url(#dqah)"/>
    <text x="340" y="58" font-size="12" font-weight="600" fill="#555">Pass</text>
    <rect x="422" y="30" width="130" height="42" rx="5" fill="#C8F7C5" stroke="#5CB85C" stroke-width="2"/>
    <text x="487" y="56" text-anchor="middle" font-size="13" font-weight="600" fill="#2D6A2E">Keep Record</text>
    <line x1="552" y1="51" x2="740" y2="100" stroke="#333" stroke-width="2" marker-end="url(#dqah)"/>
  </g>
  <!-- Continue Processing (always visible) -->
  <g class="dq-path-continue">
    <rect x="742" y="85" width="160" height="50" rx="5" fill="#C8F7C5" stroke="#5CB85C" stroke-width="2"/>
    <text x="822" y="108" text-anchor="middle" font-size="13" font-weight="600" fill="#2D6A2E">Continue</text>
    <text x="822" y="125" text-anchor="middle" font-size="13" font-weight="600" fill="#2D6A2E">Processing</text>
  </g>
  <!-- WARN path -->
  <g class="dq-path-warn">
    <line x1="468" y1="173" x2="540" y2="140" stroke="#333" stroke-width="2" marker-end="url(#dqah)"/>
    <rect x="542" y="120" width="80" height="36" rx="5" fill="#FFE0B2" stroke="#E5A100" stroke-width="2"/>
    <text x="582" y="143" text-anchor="middle" font-size="12" font-weight="600" fill="#7A5800">Warn</text>
    <path d="M 582 120 Q 582 80, 530 60" fill="none" stroke="#888" stroke-width="2" stroke-dasharray="6,4" marker-end="url(#dqahd)"/>
  </g>
  <!-- DROP path -->
  <g class="dq-path-drop">
    <line x1="490" y1="200" x2="540" y2="200" stroke="#333" stroke-width="2" marker-end="url(#dqah)"/>
    <rect x="542" y="182" width="80" height="36" rx="5" fill="#FFCDD2" stroke="#E57373" stroke-width="2"/>
    <text x="582" y="205" text-anchor="middle" font-size="12" font-weight="600" fill="#C62828">Drop</text>
    <path d="M 622 200 Q 680 180, 740 140" fill="none" stroke="#888" stroke-width="2" stroke-dasharray="6,4" marker-end="url(#dqahd)"/>
  </g>
  <!-- FAIL path -->
  <g class="dq-path-fail">
    <line x1="440" y1="245" x2="440" y2="268" stroke="#333" stroke-width="2" marker-end="url(#dqah)"/>
    <rect x="380" y="270" width="120" height="36" rx="5" fill="#FFCDD2" stroke="#D32F2F" stroke-width="2"/>
    <text x="440" y="293" text-anchor="middle" font-size="12" font-weight="600" fill="#B71C1C">Fail Flow</text>
  </g>
</svg>
<div style="font-size: 14pt; color: #666; text-align: center; margin-top: 8px; font-style: italic;">Source: <a href="https://docs.databricks.com/aws/en/ldp/expectations" style="color: #2574B5;">Databricks Expectations Documentation</a></div>
</div>

<script>
var dqActive = null;
function dqFlip(card, mode) {
  var wasFlipped = card.classList.contains('dq-flipped');
  // Reset all cards
  document.querySelectorAll('.dq-flip-card').forEach(function(c) { c.classList.remove('dq-flipped'); });
  // Toggle clicked card
  if (!wasFlipped) { card.classList.add('dq-flipped'); dqActive = mode; } else { dqActive = null; }
  // Highlight SVG paths
  var paths = ['warn','drop','fail'];
  if (dqActive) {
    paths.forEach(function(p) {
      var el = document.querySelector('.dq-path-' + p);
      if (el) el.style.opacity = (p === dqActive) ? '1' : '0.12';
    });
    // Keep shared, pass, continue visible
    document.querySelector('.dq-path-shared').style.opacity = '1';
    document.querySelector('.dq-path-pass').style.opacity = (dqActive === 'warn') ? '1' : '0.12';
    document.querySelector('.dq-path-continue').style.opacity = (dqActive !== 'fail') ? '1' : '0.12';
  } else {
    // Reset all to full opacity
    document.querySelectorAll('[class^="dq-path-"]').forEach(function(g) { g.style.opacity = '1'; });
  }
}
</script>

<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">How Expectations Work</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Inline validation</strong>: expectations are evaluated per record as data flows through the pipeline, not as a separate step after ingestion</li>
          <li style="font-size: 14pt;">The <code style="font-size: 14pt;">EXPECT</code> clause takes any <strong>deterministic SQL Boolean expression</strong>: column checks, range validations, <code style="font-size: 14pt;">IS NOT NULL</code>, date comparisons, regex patterns, or combinations with <code style="font-size: 14pt;">AND</code> / <code style="font-size: 14pt;">OR</code></li>
          <li style="font-size: 14pt;">You can stack <strong style="font-size: 14pt;">multiple constraints</strong> on a single table. Each is evaluated independently; a row can fail one constraint and pass another</li>
          <li style="font-size: 14pt;">Constraints are <strong style="font-size: 14pt;">not</strong> allowed in <code style="font-size: 14pt;">@append_flow</code> definitions: define them on the target streaming table instead</li>
        </ul>
        <strong style="color: #1B5162;">Choosing the Right Violation Action</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Warn (default)</strong>: the gentlest option. Bad records are written to the table, but violation counts are logged. Use this for <strong>observability without enforcement</strong>: you want to see what is wrong before deciding how to handle it</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">DROP ROW</strong>: the practical production choice. Bad records are silently removed and violation metrics are tracked. Use this when you can tolerate losing a few bad rows (e.g., null customer IDs on an orders table)</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">FAIL UPDATE</strong>: the strictest option. The entire refresh fails on the first bad record. Use this for <strong>regulatory or compliance gates</strong> where any invalid data is unacceptable; for example, dates that must fall within a valid range for financial reporting</li>
        </ul>
        <strong style="color: #1B5162;">Monitoring Expectations</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Violation metrics for <strong style="font-size: 14pt;">Warn</strong> and <strong style="font-size: 14pt;">DROP ROW</strong> actions are automatically logged to the pipeline event log and visible in the <strong style="font-size: 14pt;">Data Quality</strong> tab of the pipeline UI</li>
          <li style="font-size: 14pt;">You can query the event log directly to build dashboards or configure <strong style="font-size: 14pt;">SQL Alerts</strong> on violation counts; for example, alert when more than 5% of records are dropped</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">FAIL UPDATE</strong> does not log violation metrics (the pipeline fails before metrics can be captured). Use the error message to diagnose the failing record</li>
        </ul>
        <strong style="color: #1B5162;">Expression Rules and Limitations</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Expressions must be <strong style="font-size: 14pt;">deterministic</strong>: built-in SQL functions and operators only. No aggregate functions (<code style="font-size: 14pt;">SUM</code>, <code style="font-size: 14pt;">COUNT</code>), no window functions, no subqueries, no custom Python UDFs</li>
          <li style="font-size: 14pt;">This means expectations validate <strong style="font-size: 14pt;">individual rows</strong>, not cross-row patterns. For cross-row checks (primary key uniqueness, row count validation, missing record detection), use the <strong style="font-size: 14pt;">advanced expectation patterns</strong> documented in the Databricks expectation patterns guide</li>
          <li style="font-size: 14pt;">Expectations (the <code style="font-size: 14pt;">CONSTRAINT ... EXPECT</code> syntax) are supported on <strong style="font-size: 14pt;">both Streaming Tables and Materialized Views</strong> created in Databricks SQL. You declare the same constraints, with the same violation actions, on either object type</li>
          <li style="font-size: 14pt;">One caveat for Materialized Views: an MV whose definition includes expectations is <strong style="font-size: 14pt;">fully refreshed on each update</strong> and cannot refresh incrementally. If incremental refresh matters for a large MV, validate upstream and keep the MV constraint-free, or accept the full refresh as the cost of inline validation</li>
          <li style="font-size: 14pt;">MV support for expectations is recent (GA June 2026). Older material may state that only Streaming Tables support expectations; that is no longer accurate</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>


<div style="width: 100%; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="../Includes/images/icons/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0B2026; line-height: 1.7; margin-bottom: 16px;">
    Want to explore quality expectations further? Ask Genie Code. Click on the genie icon <img src="../Includes/images/icons/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0B2026;">
    <span id="genie-query-b3" style="flex: 1;">What types of quality expectations (CONSTRAINT ... EXPECT) can I add to a streaming table or a materialized view? Show me examples of the default warn behavior, ON VIOLATION DROP ROW, and ON VIOLATION FAIL UPDATE.</span>
    <button onclick="
      var text = document.getElementById('genie-query-b3').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

## C. Materialized Views: Incremental Transformations

<p style="font-size: 14pt; line-height: 1.6; color: #333;">Now that data is landing in Bronze via Streaming Tables, how do you transform it for analytics? Materialized Views are the declarative answer: pre-computed, incrementally refreshed, and serverless.</p>

### C1. What Is a Materialized View?

<p style="font-size: 14pt; line-height: 1.6; color: #333;">A <strong style="font-size: 14pt;">Materialized View</strong> is a Unity Catalog managed table that <strong>physically stores pre-computed query results</strong>: unlike a regular view, it doesn&#39;t recompute on every query.</p>

<!-- ── Visual: button-expand-panels ── -->

#### Select each feature to explore how Materialized Views work

<br/>

<div style="max-width: 100%; margin: 0 auto; font-family: sans-serif">

<style>
.mv1-row { display: block; white-space: nowrap; font-size: 0; width: 100%; box-sizing: border-box; }
.mv1-box { display: inline-block; width: 23.5%; margin-right: 2%; min-height: 80px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 12px 8px; text-align: center; box-sizing: border-box; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; }
.mv1-box:last-child { margin-right: 0; }
.mv1-box:hover { transform: translateY(-2px); }
.mv1-box.mv1-active { background: #fff; border-left-color: var(--mv1-c); border-right-color: var(--mv1-c); border-bottom-color: var(--mv1-c); }
.mv1-label { display: block; font-size: 14pt; font-weight: 700; color: #0b2026; line-height: 1.3; white-space: normal; pointer-events: none; }
.mv1-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; }
.mv1-detail-wrap.mv1-open { max-height: 1200px; opacity: 1; margin-top: 12px; }
.mv1-detail-card { background: #F9F7F4; border-radius: 10px; padding: 20px; box-sizing: border-box; border-top: 7px solid #ccc; }
.mv1-can-block { border-left: 4px solid; border-radius: 0 6px 6px 0; padding: 14px 14px; margin-bottom: 14px; font-size: 14pt; line-height: 1.6; }
.mv1-can-block ul { margin: 8px 0 0 0; padding-left: 20px; }
.mv1-can-block li { margin-bottom: 8px; }
.mv1-example-block { border-left: 4px solid; border-radius: 0 6px 6px 0; padding: 12px 14px; font-size: 14pt; line-height: 1.5; }
.mv1-example-block pre { background: #272822; color: #f8f8f2; border-radius: 6px; padding: 12px 14px; font-family: monospace; font-size: 14pt; line-height: 1.6; margin: 8px 0 0 0; white-space: pre; overflow-x: auto; }
</style>

<div style="background: #F8F9FC; border: 3px solid #1B5162; border-radius: 10px; padding: 24px; box-sizing: border-box">

  <div class="mv1-row">
    <span class="mv1-box mv1-active" data-mv1-id="0" onclick="mv1Sel(0)" style="--mv1-c:#2574B5; border-top-color:#2574B5">
      <span class="mv1-label"> The SQL</span>
    </span><span class="mv1-box" data-mv1-id="1" onclick="mv1Sel(1)" style="--mv1-c:#1B5162; border-top-color:#1B5162">
      <span class="mv1-label"> Incremental Refresh</span>
    </span><span class="mv1-box" data-mv1-id="2" onclick="mv1Sel(2)" style="--mv1-c:#FF5F46; border-top-color:#FF5F46">
      <span class="mv1-label"> Pre-Computed Results</span>
    </span><span class="mv1-box" data-mv1-id="3" onclick="mv1Sel(3)" style="--mv1-c:#FFAB00; border-top-color:#FFAB00">
      <span class="mv1-label"> Serverless</span>
    </span>
  </div>

  <div class="mv1-detail-wrap" id="mv1-detail-wrap">
    <div class="mv1-detail-card" id="mv1-detail-card"></div>
  </div>

</div>

</div>

<script>
var MV1_ITEMS = [
  { title: "The SQL You Write", color: "#2574B5", bg: "rgba(37,116,181,0.10)",
    content: `<strong style="font-size: 14pt;">One SQL statement</strong> creates a pre-computed, incrementally-refreshed table:<ul><li style="font-size: 14pt;"><code style="font-size: 14pt;">MATERIALIZED VIEW</code>: a UC managed table that stores query results physically</li><li style="font-size: 14pt;">Write standard <strong style="font-size: 14pt;">batch SQL</strong> (SELECT, JOIN, GROUP BY, window functions). The platform incrementalizes it for you</li><li style="font-size: 14pt;"><code style="font-size: 14pt;">CREATE OR REPLACE</code> executes <strong>synchronously</strong>: blocks until the initial data load completes</li><li style="font-size: 14pt;">Ideal for <strong style="font-size: 14pt;">Silver-to-Gold transformations</strong>: aggregations, joins, denormalization, business logic</li></ul>`,
    example: `<strong style="font-size: 14pt;">Create a Materialized View:</strong><pre style="background:#272822;color:#f8f8f2;border-radius:6px;padding:12px 14px;font-family:monospace; line-height:1.6;margin:8px 0 0 0;white-space:pre;overflow-x:auto"><span style="font-size: 14pt; color:#66d9ef;font-weight:600">CREATE OR REPLACE MATERIALIZED VIEW</span>\n  gold_customer_summary\n<span style="font-size: 14pt; color:#66d9ef;font-weight:600">AS</span>\n<span style="font-size: 14pt; color:#66d9ef;font-weight:600">SELECT</span>\n  c.customer_id, c.loyalty_tier,\n  <span style="font-size: 14pt; color:#a6e22e">COUNT</span>(o.order_id) <span style="font-size: 14pt; color:#66d9ef">AS</span> order_count,\n  <span style="font-size: 14pt; color:#a6e22e">SUM</span>(o.quantity * o.unit_price) <span style="font-size: 14pt; color:#66d9ef">AS</span> total_revenue\n<span style="font-size: 14pt; color:#66d9ef;font-weight:600">FROM</span> silver_orders o\n<span style="font-size: 14pt; color:#66d9ef;font-weight:600">JOIN</span> silver_customers c\n  <span style="font-size: 14pt; color:#66d9ef;font-weight:600">ON</span> o.customer_id = c.customer_id\n<span style="font-size: 14pt; color:#66d9ef;font-weight:600">GROUP BY</span> c.customer_id, c.loyalty_tier;</pre>` },
  { title: "Incremental Refresh", color: "#1B5162", bg: "rgba(2,163,111,0.10)",
    content: `Instead of recomputing the entire result from scratch, an MV detects what changed in source data and <strong>processes only the differences</strong>.<ul><li style="font-size: 14pt;"><strong style="font-size: 14pt;">You don&#39;t write incremental logic</strong>: you write standard batch SQL and the platform incrementalizes it for you</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Supports compositions</strong> of <strong>joins, aggregations, and window functions</strong>: see the full list of supported operations &#x2197; (<a style="font-size: 14pt;" href="https://docs.databricks.com/aws/en/optimizations/incremental-refresh#support-for-materialized-view-incremental-refresh">AWS</a> | <a style="font-size: 14pt;" href="https://learn.microsoft.com/en-us/azure/databricks/optimizations/incremental-refresh#support-for-materialized-view-incremental-refresh">Azure</a> | <a style="font-size: 14pt;" href="https://docs.databricks.com/gcp/en/optimizations/incremental-refresh#support-for-materialized-view-incremental-refresh">GCP</a>)</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Automatically chooses the fastest strategy</strong>: whether that means updating individual rows, overwriting partitions, recomputing aggregates, or appending new results</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Enable these source table properties</strong>: incremental refresh relies on change tracking. Without them, the platform falls back to full recompute</li></ul>`,
    example: `<strong style="font-size: 14pt;">For optimal incremental refresh</strong>, enable these on source tables:<pre style="background:#272822;color:#f8f8f2;border-radius:6px;padding:12px 14px;font-family:monospace; line-height:1.6;margin:8px 0 0 0;white-space:pre;overflow-x:auto"><span style="font-size: 14pt; color:#66d9ef;font-weight:600">ALTER TABLE</span> silver_orders <span style="font-size: 14pt; color:#66d9ef;font-weight:600">SET TBLPROPERTIES</span> (\n  <span style="font-size: 14pt; color:#e6db74">&#39;delta.enableChangeDataFeed&#39;</span> = <span style="font-size: 14pt; color:#e6db74">&#39;true&#39;</span>,\n  <span style="font-size: 14pt; color:#e6db74">&#39;delta.enableRowTracking&#39;</span> = <span style="font-size: 14pt; color:#e6db74">&#39;true&#39;</span>,\n  <span style="font-size: 14pt; color:#e6db74">&#39;delta.enableDeletionVectors&#39;</span> = <span style="font-size: 14pt; color:#e6db74">&#39;true&#39;</span>\n);</pre>` },
  { title: "Pre-Computed Results", color: "#FF5F46", bg: "rgba(255,95,70,0.10)",
    content: `A regular view recomputes on every query. An MV <strong>stores results as a physical table</strong>: queries read from that table instead of scanning raw sources.<ul><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Regular VIEW</strong>: recomputes the full query every time. 10 dashboard refreshes = 10 full scans</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Materialized VIEW</strong>: reads stored results. Cost = periodic refresh + minimal read per query</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Best for dashboards and BI</strong>: any query that runs repeatedly against large tables benefits from pre-computation. Replace the view backing your dashboard and queries go from minutes to seconds</li></ul><div style="text-align:center;padding:10px 16px;background:#fff;border:1px solid #e0e0e0;border-radius:6px; margin-top:14px"><strong style="font-size: 14pt;">Breakeven:</strong> In typical scenarios, after just a few queries between refreshes the MV pays for itself</div>`,
    example: `<strong style="font-size: 14pt;">Real-world impact:</strong><ul style="font-size: 14pt; margin:8px 0 0 0;padding-left:20px"><li style="font-size: 14pt;">&#x25C6; <a href="https://www.databricks.com/customers/anyclip" style="color:#FF5F46;font-weight:600">AnyClip</a>: replacing static views with MVs delivered a <strong style="font-size: 14pt;">98% improvement in query performance</strong>. Queries over terabytes dropped from hours to 0.5&ndash;3 minutes.</li><li style="font-size: 14pt;">&#x25C6; <a href="https://www.databricks.com/customers/comcast" style="color:#FF5F46;font-weight:600">Comcast</a>: pre-computing results from billions of interactions achieved a <strong style="font-size: 14pt;">10x reduction in compute costs</strong>.</li></ul>` },
  { title: "Serverless Execution", color: "#FFAB00", bg: "rgba(255,171,0,0.10)",
    content: `All MV refresh compute runs on <strong style="font-size: 14pt;">serverless infrastructure</strong>: the platform provisions resources, executes the refresh, and scales automatically.<li style="font-size: 14pt;"><strong style="font-size: 14pt;">Automatic scaling</strong>: the serverless pipeline <strong>right-sizes compute</strong> for each refresh</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Zero infrastructure</strong>: <strong>no clusters</strong> to configure, start, or monitor</li><li style="font-size: 14pt;"><strong style="font-size: 14pt;">Unity Catalog governed</strong>: every MV is a UC managed table with <strong>permissions, tags, lineage, and audit logging</strong></li></ul>`,
    example: `<strong style="font-size: 14pt;">Bottom line:</strong> You write a SELECT statement. The platform stores results, refreshes incrementally, scales compute, and governs the output: all serverless.` }
];
var mv1Current = null;
function mv1Sel(id) {
  var wrap = document.getElementById("mv1-detail-wrap");
  var card = document.getElementById("mv1-detail-card");
  var c = MV1_ITEMS[id];
  document.querySelectorAll(".mv1-box").forEach(function(b) { b.classList.toggle("mv1-active", parseInt(b.dataset.mv1Id,10)===id); });
  if (mv1Current === id) { wrap.classList.remove("mv1-open"); document.querySelectorAll(".mv1-box").forEach(function(b){b.classList.remove("mv1-active");}); mv1Current=null; return; }
  mv1Current = id;
  card.style.borderTopColor = c.color;
  card.innerHTML = '<div style="font-size:18pt;font-weight:700;margin-bottom:12px;color:#0b2026;">'+c.title+'</div><div class="mv1-can-block" style="background:'+c.bg+';border-color:'+c.color+'">'+c.content+'</div><div class="mv1-example-block" style="background:'+c.bg+';border-color:'+c.color+'">'+c.example+'</div>';
  wrap.classList.add("mv1-open");
}
mv1Sel(0);
[500, 1500].forEach(function(delay) {
    setTimeout(function() {
        document.querySelectorAll('.mv1-box img').forEach(function(img) {
            if (!img.complete || img.naturalWidth === 0) {
                img.src = img.src;
            }
        });
    }, delay);
});
</script>

<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">The SQL You Write</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">A <strong style="font-size: 14pt;">Materialized View</strong> is a Unity Catalog managed table that physically stores pre-computed query results. It appears in Catalog Explorer with full governance, lineage, and permissions</li>
          <li style="font-size: 14pt;">You write standard batch SQL. <code style="font-size: 14pt;">SELECT</code>, <code style="font-size: 14pt;">JOIN</code>, <code style="font-size: 14pt;">GROUP BY</code>, window functions; the engine incrementalizes it for you. No incremental logic needed</li>
          <li style="font-size: 14pt;">When you run <code style="font-size: 14pt;">CREATE OR REPLACE MATERIALIZED VIEW</code>, the data is loaded immediately. The MV is populated and ready to query as soon as the command finishes</li>
          <li style="font-size: 14pt;">Ideal for Silver-to-Gold transformations: aggregations, joins, denormalization, and business logic that powers dashboards, Genie Agents, and BI tools</li>
          <li style="font-size: 14pt;">The MV <strong style="font-size: 14pt;">stores a physical copy of the results</strong>: this is the trade-off: more storage for faster queries. It is not a cached view; it is a real table</li>
        </ul>
        <strong style="color: #1B5162;">Incremental Refresh: How It Works</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Think of an MV as a <strong style="font-size: 14pt;">pre-built report that sits on a shelf</strong>: you don&#39;t rebuild it from scratch each time someone asks for it; you just update the pages that changed</li>
          <li style="font-size: 14pt;">The engine (called <strong style="font-size: 14pt;">Enzyme</strong>) detects what changed in source tables and processes only the differences, automatically choosing the fastest strategy (updating rows, overwriting partitions, recomputing aggregates, or appending)</li>
          <li style="font-size: 14pt;"><code style="font-size: 14pt;">AUTO</code> is the default refresh policy: the platform decides the cheapest way to refresh. Small changes get incremental; large changes may trigger a full recompute</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">What supports incremental refresh:</strong> all JOIN types, GROUP BY, UNION ALL, WHERE/HAVING/QUALIFY, window functions with PARTITION BY, CTEs, and subqueries</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">What does NOT:</strong> non-deterministic functions (<code style="font-size: 14pt;">UUID()</code>, <code style="font-size: 14pt;">RANDOM()</code>), data sources with row filters or column masks, foreign catalogs, and volumes</li>
          <li style="font-size: 14pt;">For optimal incremental refresh, enable <code style="font-size: 14pt;">delta.enableChangeDataFeed</code>, <code style="font-size: 14pt;">delta.enableRowTracking</code>, and <code style="font-size: 14pt;">delta.enableDeletionVectors</code> on source tables. Without these, the platform falls back to full recompute</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Incremental refresh requires serverless compute</strong>: classic compute always does full recompute regardless of policy</li>
        </ul>
        <strong style="color: #1B5162;">Pre-Computed Results vs Regular Views</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">A regular <code style="font-size: 14pt;">VIEW</code> recomputes the full query on every access. 10 dashboard refreshes = 10 full scans of your source tables</li>
          <li style="font-size: 14pt;">A <strong style="font-size: 14pt;">Materialized View</strong> reads stored Delta results: cost = periodic refresh + minimal read per query. The breakeven is typically just a few queries</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">You can query an MV while it&#39;s refreshing</strong>: queries see the last-completed refresh state until the new refresh finishes. No downtime</li>
          <li style="font-size: 14pt;">&#x25C6; At <a style="font-size: 14pt;" href="https://www.databricks.com/customers/anyclip">AnyClip</a>, replacing static views with MVs delivered a <strong style="font-size: 14pt;">98% improvement in query performance</strong>: queries over terabytes dropped from hours to 0.5&ndash;3 minutes</li>
          <li style="font-size: 14pt;">&#x25C6; At <a style="font-size: 14pt;" href="https://www.databricks.com/customers/comcast">Comcast</a>, pre-computing results from billions of interactions achieved a <strong style="font-size: 14pt;">10x reduction in compute costs</strong></li>
        </ul>
        <strong style="color: #1B5162;">Serverless Execution</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">All MV compute runs on a <strong style="font-size: 14pt;">serverless SDP pipeline</strong>: the SQL warehouse dispatches the command, the pipeline does the heavy lifting independently</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Separate compute</strong>: refresh processing runs on the pipeline, not on your warehouse</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Automatic scaling</strong>: the serverless pipeline right-sizes compute for each refresh, no cluster configuration needed</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Unity Catalog governed</strong>: every MV is a UC managed table with permissions, tags, lineage, and audit logging from the moment it is created</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Scheduled refresh</strong> creates a managed Lakeflow Job automatically, visible in the Jobs &amp; Pipelines UI. You can also use <code style="font-size: 14pt;">TRIGGER ON UPDATE</code> for event-driven refresh</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>
<script>  
<!-- Retry any images that failed to render on initial load (500ms and 1500ms delays -->
[500, 1500].forEach(function(delay) {
    setTimeout(function() {
        document.querySelectorAll('.two-pane img').forEach(function(img) {
            if (!img.complete || img.naturalWidth === 0) {                                                                                                                                
                img.src = img.src;
            }                                                                                                                                                                             
        });     
    }, delay);
});
</script>

### C2. View vs. MV vs. Table. The Analytics Engineer's Decision

<p style="font-size: 14pt; line-height: 1.6; color: #333;">This is the decision you will make dozens of times when building a data model. Each choice is valid: the skill is matching the right type to the requirement.</p>

<!-- ── Visual: three-column decision cards ── -->
<div style="max-width: 100%; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif">
<div style="display: flex; gap: 16px; flex-wrap: wrap;">

  <!-- VIEW card -->
  <div style="flex: 1; min-width: 280px; border: 2px solid #607d8b; border-radius: 12px; overflow: hidden;">
    <div style="background: #607d8b; color: white; padding: 14px 18px; text-align: center;">
      <div style="font-size: 16pt; font-weight: 700;">Use a VIEW when</div>
    </div>
    <div style="padding: 16px 18px; background: #FAFAFA; font-size: 14pt; line-height: 1.7; color: #333;">
      <ul style="font-size: 14pt; margin: 0; padding-left: 20px;">
        <li style="font-size: 14pt;">Transforms are <strong style="font-size: 14pt;">cheap</strong> (filter, rename, cast)</li>
        <li style="font-size: 14pt;">Data changes <strong style="font-size: 14pt;">rarely</strong></li>
        <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Few consumers</strong> query the result</li>
        <li style="font-size: 14pt;">You need <strong style="font-size: 14pt;">always-current</strong> results (zero staleness)</li>
        <li style="font-size: 14pt;">You want <strong style="font-size: 14pt;">zero storage cost</strong></li>
      </ul>
      <div style="background: #272822; border-radius: 6px; padding: 10px 12px; font-family: monospace; font-size: 14pt; line-height: 1.6; color: #f8f8f2; margin-top: 12px;">
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">CREATE VIEW</span> silver_orders_clean<br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">AS SELECT</span> order_id,<br/>
&nbsp;&nbsp;<span style="font-size: 14pt; color:#a6e22e">CAST</span>(order_date <span style="font-size: 14pt; color:#66d9ef">AS</span> DATE),<br/>
&nbsp;&nbsp;quantity * unit_price <span style="font-size: 14pt; color:#66d9ef">AS</span> amount<br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">FROM</span> bronze_orders<br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">WHERE</span> status != 'cancelled';
      </div>
    </div>
  </div>

  <!-- MV card -->
  <div style="flex: 1; min-width: 280px; border: 2px solid #1B5162; border-radius: 12px; overflow: hidden;">
    <div style="background: #1B5162; color: white; padding: 14px 18px; text-align: center;">
      <div style="font-size: 16pt; font-weight: 700;">Use a MATERIALIZED VIEW when</div>
    </div>
    <div style="padding: 16px 18px; background: #FAFAFA; font-size: 14pt; line-height: 1.7; color: #333;">
      <ul style="font-size: 14pt; margin: 0; padding-left: 20px;">
        <li style="font-size: 14pt;">Transforms are <strong style="font-size: 14pt;">expensive</strong> (joins, aggregations)</li>
        <li style="font-size: 14pt;">Data changes <strong style="font-size: 14pt;">often</strong></li>
        <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Many consumers</strong> (dashboards, Genie Agents)</li>
        <li style="font-size: 14pt;">You can tolerate <strong style="font-size: 14pt;">minutes-to-hours</strong> of staleness</li>
        <li style="font-size: 14pt;">You want the platform to <strong style="font-size: 14pt;">incrementalize</strong> for you</li>
      </ul>
      <div style="background: #272822; border-radius: 6px; padding: 10px 12px; font-family: monospace; font-size: 14pt; line-height: 1.6; color: #f8f8f2; margin-top: 12px;">
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">CREATE MATERIALIZED VIEW</span><br/>
&nbsp;&nbsp;gold_customer_summary<br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">AS SELECT</span> c.customer_id,<br/>
&nbsp;&nbsp;<span style="font-size: 14pt; color:#a6e22e">COUNT</span>(o.order_id) <span style="font-size: 14pt; color:#66d9ef">AS</span> orders,<br/>
&nbsp;&nbsp;<span style="font-size: 14pt; color:#a6e22e">SUM</span>(o.amount) <span style="font-size: 14pt; color:#66d9ef">AS</span> revenue<br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">FROM</span> silver_orders o<br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">JOIN</span> silver_customers c<br/>
&nbsp;&nbsp;<span style="font-size: 14pt; color:#66d9ef;font-weight:600">ON</span> o.customer_id = c.customer_id<br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">GROUP BY</span> c.customer_id;
      </div>
    </div>
  </div>

  <!-- TABLE card -->
  <div style="flex: 1; min-width: 280px; border: 2px solid #E5A100; border-radius: 12px; overflow: hidden;">
    <div style="background: #E5A100; color: white; padding: 14px 18px; text-align: center;">
      <div style="font-size: 16pt; font-weight: 700;">Use a TABLE (INSERT INTO) when</div>
    </div>
    <div style="padding: 16px 18px; background: #FAFAFA; font-size: 14pt; line-height: 1.7; color: #333;">
      <ul style="font-size: 14pt; margin: 0; padding-left: 20px;">
        <li style="font-size: 14pt;">You need <strong style="font-size: 14pt;">parameterized logic</strong></li>
        <li style="font-size: 14pt;">ForEach or <strong style="font-size: 14pt;">looping patterns</strong></li>
        <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Custom refresh control</strong> (MERGE, conditional inserts)</li>
        <li style="font-size: 14pt;">Non-deterministic transforms that <strong style="font-size: 14pt;">cannot incrementalize</strong></li>
        <li style="font-size: 14pt;">Covered in <strong style="font-size: 14pt;">Module 4</strong></li>
      </ul>
      <div style="background: #272822; border-radius: 6px; padding: 10px 12px; font-family: monospace; font-size: 14pt; line-height: 1.6; color: #f8f8f2; margin-top: 12px;">
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">INSERT INTO</span> gold_daily_report<br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">SELECT</span> report_date,<br/>
<p style="font-size: 14pt; line-height: 1.6; color: #333;">&nbsp;&nbsp;region, total_sales<br/></p>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">FROM</span> silver_sales<br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">WHERE</span> report_date =<br/>
&nbsp;&nbsp;<span style="font-size: 14pt; color:#a6e22e">CURRENT_DATE</span>() - 1;
      </div>
    </div>
  </div>

</div>
</div>

<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">The MV as a Modeling Primitive</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">A <strong style="font-size: 14pt;">Materialized View</strong> is more than a performance optimization. It is a <strong style="font-size: 14pt;">modeling choice</strong> that declares: "this query defines a business metric, and I want the platform to keep it up to date automatically."</li>
          <li style="font-size: 14pt;">You write standard batch SQL. <code style="font-size: 14pt;">SELECT</code>, <code style="font-size: 14pt;">JOIN</code>, <code style="font-size: 14pt;">GROUP BY</code>, window functions; the Enzyme engine incrementalizes it for you. No incremental logic needed.</li>
          <li style="font-size: 14pt;">The MV stores a physical copy of the results as a table. Queries read from that table instead of scanning raw sources. You can query an MV while it is refreshing; queries see the last-completed refresh state until the new refresh finishes.</li>
        </ul>
        <strong style="color: #1B5162;">Common Modeling Patterns</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Dashboard backing table:</strong> Replace the view behind your BI dashboard with an MV. Queries go from minutes to sub-seconds.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Genie Agent source:</strong> Genie queries an MV for instant responses instead of re-running expensive joins.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Denormalized Gold table:</strong> Pre-join dimension and fact tables so downstream consumers do not need to write complex SQL.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Aggregation layer:</strong> Pre-compute SUM, COUNT, AVG at different grain levels so BI tools read aggregated results.</li>
        </ul>
        <strong style="color: #1B5162;">Source Table Properties for Optimal Refresh</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">For optimal incremental refresh, enable these properties on source tables:
            <ul>
              <li style="font-size: 14pt;"><code style="font-size: 14pt;">delta.enableChangeDataFeed = 'true'</code></li>
              <li style="font-size: 14pt;"><code style="font-size: 14pt;">delta.enableRowTracking = 'true'</code></li>
              <li style="font-size: 14pt;"><code style="font-size: 14pt;">delta.enableDeletionVectors = 'true'</code></li>
            </ul>
          </li>
          <li style="font-size: 14pt;">Without these, the platform falls back to full recompute, which is still correct but more expensive.</li>
          <li style="font-size: 14pt;">Incremental refresh also requires <strong style="font-size: 14pt;">serverless compute</strong>: classic compute always does full recompute regardless of policy.</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>

### C3. Interoperability: Iceberg-Compatible Materialized Views (Public Preview)

<div style="border-left: 4px solid #1B5162; background: #f4f8fa; padding: 14px 18px; border-radius: 4px; margin: 16px 0; font-size: 14pt; line-height: 1.6;">
<strong style="color:#1B5162;">Public Preview.</strong> You can create a Materialized View in an <strong>Apache Iceberg&ndash;compatible</strong> format so external Iceberg readers (other engines and catalogs) can read it directly - useful for cross-platform interoperability without copying data.
<pre style="background:#1B3139;color:#e8e3dc;border-radius:8px;padding:12px 16px;font-size:12pt;line-height:1.5;overflow-x:auto;margin:10px 0;">CREATE MATERIALIZED VIEW my_mv USING ICEBERG AS SELECT ...;
-- refresh Iceberg metadata for external readers
REPAIR TABLE my_mv SYNC METADATA;</pre>
Applies to standalone Materialized Views created in Databricks SQL. As a Public Preview feature, availability and syntax may change - see the <a href="https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-materialized-view">CREATE MATERIALIZED VIEW</a> reference.
</div>

## D. Choosing the Right Tool

<p style="font-size: 14pt; line-height: 1.6; color: #333;">With both Streaming Tables and Materialized Views in your toolkit, the question becomes: which one should you use for a given use case? This section provides a decision framework and peels back the architectural layer they share.</p>

### D1. ST vs MV Decision Framework

<p style="font-size: 14pt; line-height: 1.6; color: #333;">The fundamental distinction is <strong style="font-size: 14pt;">data semantics</strong>. The right choice depends on three questions:</p>

<ul style="font-size: 14pt; line-height: 1.7">
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">Append-only or updates?</strong>: STs are <strong>append-by-default</strong>. MVs maintain the full result and <strong>reflect source changes</strong></li>
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">Ingestion or transformation?</strong>: STs use <code style="font-size: 14pt;">read_files()</code> for <strong>file ingestion</strong>. MVs use <code style="font-size: 14pt;">SELECT</code> for <strong>joins and aggregations</strong></li>
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">Where in the medallion?</strong>: Bronze = <strong>ST</strong> &bull; Gold = <strong>MV</strong> &bull; Silver = <strong>depends</strong></li>
</ul>

#### Click **Yes** or **No** at each step to follow your path

<!-- ── Visual: decision-tree ── -->
<div class="d1dt-wrap">
<style>
.d1dt-wrap { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 100%; margin: 0 auto; padding: 24px; background: #F9F7F4; border: 1px solid #E8E3DC; border-radius: 12px; }
.d1dt-tree { display: flex; flex-direction: column; align-items: center; gap: 0; }
.d1dt-node { background: #FFF; border: 2px solid #E8E3DC; border-radius: 10px; padding: 18px 24px; max-width: 100%; width: 100%; text-align: center; }
.d1dt-node-q { font-size: 15pt; font-weight: 700; color: #0b2026; margin: 0 0 14px; line-height: 1.4; }
.d1dt-node-active { border-color: #2574B5; box-shadow: 0 2px 12px rgba(37,116,181,0.12); }
.d1dt-buttons { display: flex; gap: 12px; justify-content: center; }
.d1dt-btn { padding: 8px 28px; font-size: 14pt; font-weight: 700; border: 2px solid; border-radius: 6px; cursor: pointer; transition: background 0.15s, color 0.15s; background: #FFF; }
.d1dt-btn-yes { color: #1B5162; border-color: #1B5162; }
.d1dt-btn-yes:hover, .d1dt-btn-yes.d1dt-chosen { background: #1B5162; color: #FFF; }
.d1dt-btn-no { color: #E0382B; border-color: #E0382B; }
.d1dt-btn-no:hover, .d1dt-btn-no.d1dt-chosen { background: #E0382B; color: #FFF; }
.d1dt-conn { width: 2px; height: 24px; background: #E8E3DC; margin: 0 auto; }
.d1dt-conn-active { background: #2574B5; }
.d1dt-branch { display: none; flex-direction: column; align-items: center; width: 100%; }
.d1dt-branch.d1dt-vis { display: flex; }
.d1dt-split { display: flex; gap: 24px; width: 100%; justify-content: center; }
.d1dt-split-col { display: flex; flex-direction: column; align-items: center; flex: 1; max-width: 420px; }
.d1dt-leaf { background: #FFF; border-radius: 10px; padding: 18px 24px; min-width: 320px; max-width: 100%; width: 100%; text-align: center; }
.d1dt-leaf-text { font-size: 14pt; font-weight: 600; line-height: 1.5; margin: 0; }
.d1dt-rec-st { border: 2px solid #1B5162; background: rgba(2,163,111,0.04); }
.d1dt-rec-st .d1dt-leaf-text { color: #1B5162; }
.d1dt-rec-st::before { content: 'Streaming Table'; display: block; font-size: 14pt; font-weight: 700; color: #1B5162; text-transform: uppercase; letter-spacing: 0.06em; margin-bottom: 8px; }
.d1dt-rec-mv { border: 2px solid #2574B5; background: rgba(37,116,181,0.04); }
.d1dt-rec-mv .d1dt-leaf-text { color: #2574B5; }
.d1dt-rec-mv::before { content: 'Materialized View'; display: block; font-size: 14pt; font-weight: 700; color: #2574B5; text-transform: uppercase; letter-spacing: 0.06em; margin-bottom: 8px; }
.d1dt-alt { border: 2px solid #E5A100; background: rgba(229,161,0,0.04); }
.d1dt-alt .d1dt-leaf-text { color: #E5A100; }
.d1dt-alt::before { content: 'Alternative'; display: block; font-size: 14pt; font-weight: 700; color: #E5A100; text-transform: uppercase; letter-spacing: 0.06em; margin-bottom: 8px; }
.d1dt-muted { opacity: 0.35; pointer-events: none; }
.d1dt-reset { display: inline-block; margin-top: 20px; padding: 8px 20px; font-size: 14pt; font-weight: 600; color: #1B3139; background: #FFF; border: 1px solid #E8E3DC; border-radius: 6px; cursor: pointer; }
.d1dt-reset:hover { background: #F9F7F4; }
</style>
<div class="d1dt-tree" id="d1dt-tree">
  <div class="d1dt-node d1dt-node-active" id="d1dt-root">
    <p class="d1dt-node-q">Is this ingestion from external cloud storage?</p>
    <div class="d1dt-buttons">
      <button class="d1dt-btn d1dt-btn-yes" onclick="d1dtC('root','yes')">Yes</button>
      <button class="d1dt-btn d1dt-btn-no" onclick="d1dtC('root','no')">No</button>
    </div>
  </div>
  <div class="d1dt-split">
    <div class="d1dt-split-col">
      <div class="d1dt-conn" id="d1dt-conn-root-yes"></div>
      <div class="d1dt-branch" id="d1dt-branch-root-yes">
        <div class="d1dt-node" id="d1dt-node-y1">
          <p class="d1dt-node-q">Is the source append-only (logs, events, sensors)?</p>
          <div class="d1dt-buttons">
            <button class="d1dt-btn d1dt-btn-yes" onclick="d1dtC('y1','yes')">Yes</button>
            <button class="d1dt-btn d1dt-btn-no" onclick="d1dtC('y1','no')">No</button>
          </div>
        </div>
        <div class="d1dt-split">
          <div class="d1dt-split-col">
            <div class="d1dt-conn" id="d1dt-conn-y1-yes"></div>
            <div class="d1dt-branch" id="d1dt-branch-y1-yes">
              <div class="d1dt-leaf d1dt-rec-st"><p class="d1dt-leaf-text">Streaming Table: incremental ingestion with Auto Loader</p></div>
            </div>
          </div>
          <div class="d1dt-split-col">
            <div class="d1dt-conn" id="d1dt-conn-y1-no"></div>
            <div class="d1dt-branch" id="d1dt-branch-y1-no">
              <div class="d1dt-leaf d1dt-rec-st"><p class="d1dt-leaf-text">Streaming Table with AUTO CDC: handles inserts, updates, and deletes (Lecture 3)</p></div>
            </div>
          </div>
        </div>
      </div>
    </div>
    <div class="d1dt-split-col">
      <div class="d1dt-conn" id="d1dt-conn-root-no"></div>
      <div class="d1dt-branch" id="d1dt-branch-root-no">
        <div class="d1dt-node" id="d1dt-node-n1">
          <p class="d1dt-node-q">Do you need aggregations, joins, or business logic?</p>
          <div class="d1dt-buttons">
            <button class="d1dt-btn d1dt-btn-yes" onclick="d1dtC('n1','yes')">Yes</button>
            <button class="d1dt-btn d1dt-btn-no" onclick="d1dtC('n1','no')">No</button>
          </div>
        </div>
        <div class="d1dt-split">
          <div class="d1dt-split-col">
            <div class="d1dt-conn" id="d1dt-conn-n1-yes"></div>
            <div class="d1dt-branch" id="d1dt-branch-n1-yes">
              <div class="d1dt-leaf d1dt-rec-mv"><p class="d1dt-leaf-text">Materialized View: the platform incrementalizes your SQL automatically</p></div>
            </div>
          </div>
          <div class="d1dt-split-col">
            <div class="d1dt-conn" id="d1dt-conn-n1-no"></div>
            <div class="d1dt-branch" id="d1dt-branch-n1-no">
              <div class="d1dt-node" id="d1dt-node-n1n1">
                <p class="d1dt-node-q">Does source data get updated or deleted?</p>
                <div class="d1dt-buttons">
                  <button class="d1dt-btn d1dt-btn-yes" onclick="d1dtC('n1n1','yes')">Yes</button>
                  <button class="d1dt-btn d1dt-btn-no" onclick="d1dtC('n1n1','no')">No</button>
                </div>
              </div>
              <div class="d1dt-split">
                <div class="d1dt-split-col">
                  <div class="d1dt-conn" id="d1dt-conn-n1n1-yes"></div>
                  <div class="d1dt-branch" id="d1dt-branch-n1n1-yes">
                    <div class="d1dt-leaf d1dt-rec-mv"><p class="d1dt-leaf-text">Materialized View: reflects updates/deletes from source</p></div>
                  </div>
                </div>
                <div class="d1dt-split-col">
                  <div class="d1dt-conn" id="d1dt-conn-n1n1-no"></div>
                  <div class="d1dt-branch" id="d1dt-branch-n1n1-no">
                    <div class="d1dt-leaf d1dt-alt"><p class="d1dt-leaf-text">Either works: ST for simple append pass-through, MV if you may need to reprocess later</p></div>
                  </div>
                </div>
              </div>
            </div>
          </div>
        </div>
      </div>
    </div>
  </div>
  <button class="d1dt-reset" onclick="d1dtReset()">Reset</button>
</div>
<script>
function d1dtC(n,c){var y=document.getElementById('d1dt-branch-'+n+'-yes'),x=document.getElementById('d1dt-branch-'+n+'-no'),yc=document.getElementById('d1dt-conn-'+n+'-yes'),xc=document.getElementById('d1dt-conn-'+n+'-no'),nd=document.getElementById('d1dt-node-'+n)||document.getElementById('d1dt-root');if(!y||!x)return;nd.classList.add('d1dt-node-active');if(c==='yes'){y.classList.add('d1dt-vis');x.classList.remove('d1dt-vis');x.classList.add('d1dt-muted');y.classList.remove('d1dt-muted');if(yc)yc.classList.add('d1dt-conn-active');if(xc)xc.classList.remove('d1dt-conn-active');nd.querySelectorAll('.d1dt-btn').forEach(function(b){b.classList.remove('d1dt-chosen')});nd.querySelector('.d1dt-btn-yes').classList.add('d1dt-chosen');}else{x.classList.add('d1dt-vis');y.classList.remove('d1dt-vis');y.classList.add('d1dt-muted');x.classList.remove('d1dt-muted');if(xc)xc.classList.add('d1dt-conn-active');if(yc)yc.classList.remove('d1dt-conn-active');nd.querySelectorAll('.d1dt-btn').forEach(function(b){b.classList.remove('d1dt-chosen')});nd.querySelector('.d1dt-btn-no').classList.add('d1dt-chosen');}}
function d1dtShowAll(){document.querySelectorAll('.d1dt-branch').forEach(function(b){b.classList.add('d1dt-vis');b.classList.remove('d1dt-muted')});document.querySelectorAll('.d1dt-conn').forEach(function(c){c.classList.add('d1dt-conn-active')});document.querySelectorAll('.d1dt-node').forEach(function(n){n.classList.add('d1dt-node-active')});document.querySelectorAll('.d1dt-btn').forEach(function(b){b.classList.remove('d1dt-chosen')});}
function d1dtReset(){document.querySelectorAll('.d1dt-branch').forEach(function(b){b.classList.remove('d1dt-vis','d1dt-muted')});document.querySelectorAll('.d1dt-conn').forEach(function(c){c.classList.remove('d1dt-conn-active')});document.querySelectorAll('.d1dt-node').forEach(function(n){n.classList.remove('d1dt-node-active')});document.getElementById('d1dt-root').classList.add('d1dt-node-active');document.querySelectorAll('.d1dt-btn').forEach(function(b){b.classList.remove('d1dt-chosen')});}
</script>
</div>

<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">Understanding the Core Difference</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Streaming Table:</strong> Each input row is processed exactly once. Once a row is appended, it is never re-queried with later updates. Think of an ST as a conveyor belt: items go on once and move through in order, never going back.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Materialized View:</strong> The entire query result is maintained and can be refreshed to reflect changes in source data. Think of an MV as a whiteboard summary: erased and updated whenever the source material changes.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Medallion architecture mapping:</strong>
            <ul>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Bronze:</strong> Streaming Table: raw ingestion from cloud storage via Auto Loader</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Silver (simple cleaning):</strong> Streaming Table: append-only transformations, filtering, type casting</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Silver (complex enrichment):</strong> Materialized View: joins, aggregations, reprocessing</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Gold:</strong> Materialized View: pre-computed aggregations, BI-ready</li>
            </ul>
          </li>
          <li style="font-size: 14pt;">Both are stored as tables in Unity Catalog and backed by serverless SDP pipelines.</li>
        </ul>
        <strong style="color: #1B5162;">Common Misconceptions</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><em>"STs are for streaming, MVs are for batch."</em> Oversimplified. STs process each row once (streaming semantics) but can be scheduled like batch. MVs use batch semantics but can be refreshed frequently or triggered on upstream changes.</li>
          <li style="font-size: 14pt;"><em>"STs can do aggregations."</em> Technically yes, with stateful streaming (watermarks and windows), but MVs are the simpler and recommended choice for aggregation use cases.</li>
          <li style="font-size: 14pt;"><em>"MVs can replace STs for ingestion."</em> No. MVs do not use Auto Loader or <code style="font-size: 14pt;">read_files()</code>. They cannot ingest from cloud storage the way STs can.</li>
          <li style="font-size: 14pt;"><em>"Use STs everywhere for best performance."</em> STs are append-only and see each row once. If you need to reflect source data changes (updates, deletes), you need an MV.</li>
        </ul>
        <strong style="color: #1B5162;">Applying the Framework</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Walk through the decision tree by asking:
            <ul>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Is this ingestion from external storage?</strong> &rarr; Streaming Table with Auto Loader</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Is the source append-only?</strong> (logs, events, sensors) &rarr; Streaming Table</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Do you need aggregations, joins, or business logic?</strong> &rarr; Materialized View</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Does source data get updated or deleted?</strong> &rarr; Materialized View (or ST with AUTO CDC for ingestion)</li>
            </ul>
          </li>
          <li style="font-size: 14pt;">For CDC processing (SCD Type 1/2), use a Streaming Table with AUTO CDC flow.</li>
          <li style="font-size: 14pt;">MVs and STs work together: a common pattern is an MV reading from an ST. ST for Bronze ingestion, MV for Gold aggregation. An ST can also stream from another ST.</li>
          <li style="font-size: 14pt;">Rule of thumb: <strong style="font-size: 14pt;">When in doubt, use MV for transformations and ST for ingestion.</strong></li>
        </ul>
        <strong style="color: #1B5162;">All Decision Paths at a Glance</strong>
        <table style="width:100%;border-collapse:collapse;margin:8px 0 0 0; line-height:1.6">
          <thead><tr style="background:#1B5162;color:#fff;">
            <th style="padding:10px 12px;text-align:center;border:1px solid #1B5162; font-size: 14pt;">Ingestion from<br/>cloud storage?</th>
            <th style="padding:10px 12px;text-align:center;border:1px solid #1B5162; font-size: 14pt;">Append-only<br/>source?</th>
            <th style="padding:10px 12px;text-align:center;border:1px solid #1B5162; font-size: 14pt;">Aggregations,<br/>joins, logic?</th>
            <th style="padding:10px 12px;text-align:center;border:1px solid #1B5162; font-size: 14pt;">Source<br/>updates/deletes?</th>
            <th style="padding:10px 12px;text-align:center;border:1px solid #1B5162; font-size: 14pt;">Result</th>
          </tr></thead>
          <tbody>
            <tr style="background:#f7fdf4;"><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#1B5162; font-size: 14pt;">Yes</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#1B5162; font-size: 14pt;">Yes</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;color:#999; font-size: 14pt;">-</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;color:#999; font-size: 14pt;">-</td><td style="padding:8px 12px;border:1px solid #E8E3DC;color:#1B5162;font-weight:700; font-size: 14pt;">Streaming Table</td></tr>
            <tr style="background:#f7fdf4;"><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#1B5162; font-size: 14pt;">Yes</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#E0382B; font-size: 14pt;">No</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;color:#999; font-size: 14pt;">-</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;color:#999; font-size: 14pt;">-</td><td style="padding:8px 12px;border:1px solid #E8E3DC;color:#1B5162;font-weight:700; font-size: 14pt;">ST + AUTO CDC</td></tr>
            <tr style="background:#f0f7ff;"><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#E0382B; font-size: 14pt;">No</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;color:#999; font-size: 14pt;">-</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#1B5162; font-size: 14pt;">Yes</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;color:#999; font-size: 14pt;">-</td><td style="padding:8px 12px;border:1px solid #E8E3DC;color:#2574B5;font-weight:700; font-size: 14pt;">Materialized View</td></tr>
            <tr style="background:#f0f7ff;"><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#E0382B; font-size: 14pt;">No</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;color:#999; font-size: 14pt;">-</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#E0382B; font-size: 14pt;">No</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#1B5162; font-size: 14pt;">Yes</td><td style="padding:8px 12px;border:1px solid #E8E3DC;color:#2574B5;font-weight:700; font-size: 14pt;">Materialized View</td></tr>
            <tr style="background:#fff8e1;"><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#E0382B; font-size: 14pt;">No</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;color:#999; font-size: 14pt;">-</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#E0382B; font-size: 14pt;">No</td><td style="padding:8px 12px;text-align:center;border:1px solid #E8E3DC;font-weight:600;color:#E0382B; font-size: 14pt;">No</td><td style="padding:8px 12px;border:1px solid #E8E3DC;color:#E5A100;font-weight:700; font-size: 14pt;">Either works</td></tr>
          </tbody>
        </table>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>

## E. Operationalizing Your Pipeline

You know what Streaming Tables and Materialized Views are, and when to use each. Now it is time to make them production-ready: validating incrementalizability before deployment, scheduling automated refreshes, and monitoring pipeline health.

### E1. EXPLAIN CREATE MATERIALIZED VIEW

<code style="font-size: 14pt;">EXPLAIN CREATE MATERIALIZED VIEW</code> analyzes your query <strong>before</strong> you create the MV, telling you whether it qualifies for incremental refresh so you can <strong style="font-size: 14pt;">fix problems before they reach production</strong>.

<!-- ── Visual: D1 integrated ── -->
<div class="d1-wrap">
<style>
/* ── D1 scoped styles ── */
.d1-wrap { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 100%; margin: 0 auto; color: #1B3139; }
/* ── Workflow top ── */
.d1-flow { display: flex; flex-direction: column; align-items: center; gap: 0; padding: 16px 0 0; }
.d1-flow-step { background: #E8F0FE; border: 2px solid #2574B5; border-radius: 10px; padding: 12px 28px; text-align: center; font-size: 14pt; font-weight: 600; }
.d1-flow-arrow { font-size: 20pt; line-height: 1; margin: 2px 0; }
.d1-flow-hint { font-size: 14pt; color: #618794; margin: 4px 0 6px; }
/* ── Flip cards as decision point ── */
.d1-fc-row { display: flex; gap: 20px; justify-content: center; perspective: 1200px; }
.d1-fc-col { display: flex; flex-direction: column; align-items: center; }
.d1-fc-col-yes { flex: 1; }
.d1-fc-col-no  { flex: 1; }
.d1-fc-card { width: 100%; height: 370px; perspective: 1000px; cursor: pointer; }
.d1-fc-inner { position: relative; width: 100%; height: 100%; transition: transform 0.6s cubic-bezier(0.4,0,0.2,1); transform-style: preserve-3d; }
.d1-fc-card.flipped .d1-fc-inner { transform: rotateY(180deg); }
.d1-fc-face { position: absolute; top: 0; left: 0; width: 100%; height: 100%; backface-visibility: hidden; -webkit-backface-visibility: hidden; border-radius: 12px; box-sizing: border-box; overflow: hidden; }
.d1-fc-front { display: flex; flex-direction: column; }
.d1-fc-front-hdr { padding: 10px 16px; text-align: center; color: white; font-size: 14pt; font-weight: 700; }
.d1-fc-front-code { flex: 1; background: #272822; color: #f8f8f2; padding: 14px 16px; font-family: monospace; font-size: 14pt; line-height: 1.7; }
.d1-fc-front-hint { text-align: center; padding: 6px; font-size: 14pt; color: #618794; background: #F9F7F4; }
.d1-fc-back { transform: rotateY(180deg); padding: 18px; display: flex; flex-direction: column; font-size: 14pt; font-family: monospace; line-height: 1.5; }
.d1-fc-back-sec { color: #618794; margin-top: 10px; }
.d1-fc-back-sec:first-child { margin-top: 0; }
/* ── Outcome boxes below each card ── */
.d1-outcome { margin-top: 6px; width: 100%; }
.d1-outcome-arrow { font-size: 18pt; line-height: 1; text-align: center; }
.d1-outcome-yes { background: #E6F9F0; border: 2px solid #00A972; border-radius: 10px; padding: 12px 16px; text-align: center; font-size: 14pt; font-weight: 600; }
.d1-outcome-no { display: flex; gap: 8px; }
.d1-outcome-no-box { flex: 1; background: #FFF0EE; border: 2px solid #FF5F46; border-radius: 10px; padding: 12px 8px; text-align: center; font-size: 14pt; font-weight: 600; }
.d1-outcome-or { display: flex; align-items: center; font-size: 14pt; font-weight: 700; color: #888; }
.d1-outcome-sub { font-size: 14pt; font-weight: 400; color: #555; margin-top: 2px; }
.d1-outcome-caveat { font-size: 14pt; color: #7a6a00; font-style: italic; margin-top: 6px; text-align: center; }
/* ── Lane backgrounds (applied via HTML classes alongside flex above) ── */
.d1-fc-col-yes.d1-fc-col-yes { background: rgba(0,169,114,0.06); border-radius: 16px; padding: 16px 14px 20px; }
.d1-fc-col-no.d1-fc-col-no   { background: rgba(255,95,70,0.06); border-radius: 16px; padding: 16px 14px 20px; }
/* ── Error grid inside NO lane ── */
.d1-err-label { font-size: 14pt; font-weight: 600; color: #FF5F46; margin: 16px 0 8px; }
.d1-err-grid { display: grid; grid-template-columns: 1fr; gap: 10px; }
.d1-err-card { background: #fff; border-radius: 8px; padding: 12px 14px; border-top: 3px solid #FF5F46; }
.d1-err-name { font-weight: 700; font-size: 14pt; color: #FF5F46; font-family: monospace; }
.d1-err-list { font-size: 14pt; color: #444; margin: 6px 0 0 0; padding-left: 18px; line-height: 1.5; }
</style>

<!-- ── STEPS 1 & 2 ── -->
<div class="d1-flow">
  <div style="font-size: 15pt; font-weight: 700; color: #1B3139; margin-bottom: 8px;">Recommended Workflow</div>
  <div class="d1-flow-step">1. Write your MV query</div>
  <div class="d1-flow-arrow" style="color:#2574B5">&#x25BC;</div>
  <div class="d1-flow-step">2. Run <code style="font-size: 14pt; background:#d0e4f7;padding:2px 6px;border-radius:4px;">EXPLAIN CREATE MATERIALIZED VIEW</code></div>
  <div class="d1-flow-arrow" style="color:#2574B5">&#x25BC;</div>
  <div class="d1-flow-hint">Click each card to see what EXPLAIN returns</div>
</div>

<!-- ── FLIP CARDS = THE DECISION POINT ── -->
<div class="d1-fc-row">
  <!-- ── YES lane ── -->
  <div class="d1-fc-col d1-fc-col-yes">
    <div class="d1-fc-card" onclick="this.classList.toggle('flipped')">
      <div class="d1-fc-inner">
        <div class="d1-fc-face d1-fc-front" style="border: 2px solid #00A972">
          <div class="d1-fc-front-hdr" style="background:#00A972"> Incrementalizable</div>
          <div class="d1-fc-front-code">
            <span style="font-size: 14pt; color:#66d9ef;font-weight:600">EXPLAIN CREATE MATERIALIZED VIEW</span><br/>
gold_revenue <span style="font-size: 14pt; color:#66d9ef">AS</span><br/>
            <span style="font-size: 14pt; color:#66d9ef;font-weight:600">SELECT</span> product,<br/>
<p style="font-size: 14pt; line-height: 1.6; color: #f8f8f2;">&nbsp;&nbsp;<span style="font-size: 14pt; color:#a6e22e">SUM</span>(quantity) <span style="font-size: 14pt; color:#66d9ef">AS</span> total_qty<br/></p>
            <span style="font-size: 14pt; color:#66d9ef;font-weight:600">FROM</span> sales_transactions<br/>
            <span style="font-size: 14pt; color:#66d9ef;font-weight:600">GROUP BY</span> product;
          </div>
          <div class="d1-fc-front-hint"> Click to see output</div>
        </div>
        <div class="d1-fc-face d1-fc-back" style="background:#0b3d2b;color:#a8e6cf;border:2px solid #00A972;border-radius:12px">
          <div class="d1-fc-back-sec" style="color:#66bb6a">== Incremental Update Eligibility ==</div>
          <div style="color:#fff;font-weight:700;margin-top:4px">The Materialized View <strong>can</strong> be incrementally refreshed.</div>
          <div class="d1-fc-back-sec" style="color:#66bb6a">== Detailed Incrementalization Info ==</div>
          <div style="color:#a8e6cf;margin-top:4px">No issues detected.</div>
          <div class="d1-fc-back-sec" style="color:#66bb6a">== Physical Plan ==</div>
          <div style="color:#a8e6cf;margin-top:4px; opacity:0.7">Execute CreateMaterializedViewCommand<br/>&nbsp;&nbsp;+- Aggregate [product], [product, sum(quantity)]</div>
        </div>
      </div>
    </div>
    <div class="d1-outcome">
      <div class="d1-outcome-arrow" style="color:#00A972">&#x25BC;</div>
      <div class="d1-outcome-yes">3. Choose your <code style="font-size: 14pt; background:#c8f0dc;padding:2px 6px;border-radius:4px;">REFRESH POLICY</code><br/><span class="d1-outcome-sub">(AUTO | INCREMENTAL | STRICT)</span></div>
      <div class="d1-outcome-caveat">Incrementalizable &#x2260; incrementalized. Under AUTO, the cost model may still pick full recompute. Use INCREMENTAL or STRICT to force it.</div>
      <!-- Full sequence example -->
      <div style="margin-top: 14px; background: #272822; border-radius: 10px; padding: 14px 16px; margin-top: 6px; font-family: monospace; line-height: 1.8; color: #f8f8f2">
        <span style="font-size: 14pt; color:#618794">-- Create with policy</span><br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">CREATE MATERIALIZED VIEW</span><br/>
gold_revenue<br/>
        <span style="font-size: 14pt; color:#a6e22e;font-weight:600">REFRESH POLICY AUTO</span><br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">AS</span><br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">SELECT</span> product,<br/>
<p style="font-size: 14pt; line-height: 1.6; color: #f8f8f2;">&nbsp;&nbsp;<span style="font-size: 14pt; color:#a6e22e">SUM</span>(quantity) <span style="font-size: 14pt; color:#66d9ef">AS</span> total_qty<br/></p>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">FROM</span> sales_transactions<br/>
        <span style="font-size: 14pt; color:#66d9ef;font-weight:600">GROUP BY</span> product;
      </div>
    </div>
  </div>
  <!-- ── NO lane ── -->
  <div class="d1-fc-col d1-fc-col-no">
    <div class="d1-fc-card" onclick="this.classList.toggle('flipped')">
      <div class="d1-fc-inner">
        <div class="d1-fc-face d1-fc-front" style="border: 2px solid #FF5F46">
          <div class="d1-fc-front-hdr" style="background:#FF5F46"> Not Incrementalizable</div>
          <div class="d1-fc-front-code">
            <span style="font-size: 14pt; color:#66d9ef;font-weight:600">EXPLAIN CREATE MATERIALIZED VIEW</span><br/>
gold_orders <span style="font-size: 14pt; color:#66d9ef">AS</span><br/>
            <span style="font-size: 14pt; color:#66d9ef;font-weight:600">SELECT</span> *,<br/>
<p style="font-size: 14pt; line-height: 1.6; color: #f8f8f2;">&nbsp;&nbsp;<span style="font-size: 14pt; color:#a6e22e">UUID</span>() <span style="font-size: 14pt; color:#66d9ef">AS</span> row_id<br/></p>
            <span style="font-size: 14pt; color:#66d9ef;font-weight:600">FROM</span> trips;
          </div>
          <div class="d1-fc-front-hint"> Click to see output</div>
        </div>
        <div class="d1-fc-face d1-fc-back" style="background:#3d1111;color:#ffab91;border:2px solid #FF5F46;border-radius:12px">
          <div class="d1-fc-back-sec" style="color:#ef5350">== Incremental Update Eligibility ==</div>
          <div style="color:#fff;font-weight:700;margin-top:4px">The Materialized View <strong>cannot</strong> be incrementally refreshed.</div>
          <div class="d1-fc-back-sec" style="color:#ef5350">== Detailed Incrementalization Info ==</div>
          <div style="color:#ffab91;margin-top:4px">- EXPRESSION_NOT_DETERMINISTIC:<br/>&nbsp;&nbsp;UUID is not deterministic.<br/>&nbsp;&nbsp;Consider rewriting the query.</div>
          <div style="color:#ffab91;margin-top:6px">- ROW_TRACKING_NOT_ENABLED:<br/>&nbsp;&nbsp;Enable row tracking on<br/>&nbsp;&nbsp;source tables.</div>
        </div>
      </div>
    </div>
    <div class="d1-outcome">
      <div class="d1-outcome-arrow" style="color:#FF5F46">&#x25BC;</div>
      <div class="d1-outcome-no">
        <div class="d1-outcome-no-box">4a. Redesign<br/>the query<br/><span class="d1-outcome-sub">then re-run EXPLAIN</span></div>
        <div class="d1-outcome-or">OR</div>
        <div class="d1-outcome-no-box">4b. Accept full<br/>recompute<br/><code style="font-size: 14pt; background:#fdd;padding:2px 6px;border-radius:4px">AUTO</code> / <code style="font-size: 14pt; background:#fdd;padding:2px 6px;border-radius:4px">FULL</code></div>
      </div>
    </div>
    <!-- Error conditions — inside the NO lane -->
    <div class="d1-err-label">Common error conditions:</div>
    <div class="d1-err-grid">
      <div class="d1-err-card">
        <div class="d1-err-name">EXPRESSION_NOT_DETERMINISTIC</div>
        <ul class="d1-err-list">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Cause:</strong> <code style="font-size: 14pt;">UUID()</code>, <code style="font-size: 14pt;">RAND()</code>: <strong>different result every run</strong></li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Fix:</strong> Remove the function or compute at <strong>query time</strong></li>
        </ul>
      </div>
      <div class="d1-err-card">
        <div class="d1-err-name">ROW_TRACKING_NOT_ENABLED</div>
        <ul class="d1-err-list">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Cause:</strong> Source tables missing <strong>change tracking</strong></li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Fix:</strong> <code style="font-size: 14pt;">ALTER TABLE SET TBLPROPERTIES('delta.enableRowTracking'='true')</code></li>
        </ul>
      </div>
    </div>
    <div style="text-align: right; margin-top: 6px">All documented error conditions &#x2197; (<a style="font-size: 14pt;" href="https://docs.databricks.com/aws/en/error-messages/materialized-view-not-incrementalizable-error-class">AWS</a> | <a style="font-size: 14pt;" href="https://learn.microsoft.com/en-us/azure/databricks/error-messages/materialized-view-not-incrementalizable-error-class">Azure</a> | <a style="font-size: 14pt;" href="https://docs.databricks.com/gcp/en/error-messages/materialized-view-not-incrementalizable-error-class">GCP</a>)</div>
  </div>
</div>

</div><!-- end d1-wrap -->

<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">What Makes a Query Non-Incrementalizable?</strong>
        <ul style="font-size: 14pt; margin: 8px 0 14px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">When you run <code style="font-size: 14pt;">EXPLAIN CREATE MATERIALIZED VIEW</code>, it checks your query for patterns that <strong style="font-size: 14pt;">block incremental refresh</strong>:
            <ul>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Non-deterministic functions:</strong> <code style="font-size: 14pt;">UUID()</code>, <code style="font-size: 14pt;">RANDOM()</code>, <code style="font-size: 14pt;">RAND()</code>: these produce different results every run, so the engine cannot track what changed</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Non-Delta sources:</strong> CSV, JSON, or foreign catalog tables (only UC managed tables support change tracking)</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Row filters or column masks:</strong> security policies on source data prevent incremental tracking</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">External UDFs:</strong> user-defined functions that call external services are treated as non-deterministic</li>
            </ul>
          </li>
          <li style="font-size: 14pt;">On the other hand, these patterns <strong style="font-size: 14pt;">are</strong> incrementalizable: you can use them freely:
            <ul>
              <li style="font-size: 14pt;"><code style="font-size: 14pt;">SELECT</code> with deterministic expressions, <code style="font-size: 14pt;">GROUP BY</code> with standard aggregations</li>
              <li style="font-size: 14pt;">All <code style="font-size: 14pt;">JOIN</code> types, <code style="font-size: 14pt;">UNION ALL</code>, <code style="font-size: 14pt;">WHERE</code>/<code style="font-size: 14pt;">HAVING</code>/<code style="font-size: 14pt;">QUALIFY</code>, CTEs</li>
              <li style="font-size: 14pt;">Window functions with <code style="font-size: 14pt;">PARTITION BY</code></li>
              <li style="font-size: 14pt;">The special exceptions <code style="font-size: 14pt;">current_date()</code> and <code style="font-size: 14pt;">current_timestamp()</code></li>
            </ul>
          </li>
          <li style="font-size: 14pt;">Think of EXPLAIN as a <strong style="font-size: 14pt;">building inspector</strong> who checks your blueprints before construction: the inspector confirms the design is structurally sound, but cannot predict the weather on construction day. Similarly, EXPLAIN confirms your query <em>can</em> be incrementalized, but the runtime cost model ultimately decides <em>whether</em> it will be.</li>
        </ul>
        <strong style="color: #1B5162;">In Practice</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">EXPLAIN consumes <strong style="font-size: 14pt;">minimal compute</strong>: it analyzes your query structure without processing any data. You can run it as many times as you want during development at negligible cost.</li>
          <li style="font-size: 14pt;">This is <strong style="font-size: 14pt;">generally available</strong>. On serverless SQL Warehouses it works without any separate enablement. You can start using it today.</li>
          <li style="font-size: 14pt;">A good habit is to run EXPLAIN on every MV query before you deploy it. It takes seconds and can save you hours of debugging unexpected full-recompute behavior in production.</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>

### E2. SQL Patterns That Incrementalize (and Those That Don't)

<p style="font-size: 14pt; line-height: 1.6; color: #333;">The way you structure your SQL determines whether Enzyme can optimize it. This is a <strong style="font-size: 14pt;">modeling skill</strong>: learning which patterns to use and which to restructure.</p>

<!-- ── Visual: incrementalizable patterns comparison ── -->
<div style="max-width: 1400px; margin: 0 auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif">

<div style="display: flex; gap: 16px; flex-wrap: wrap;">

  <!-- Incrementalizes -->
  <div style="flex: 1; min-width: 400px;">
    <div style="background: #00A972; color: white; padding: 12px 18px; border-radius: 10px 10px 0 0; text-align: center; font-size: 15pt; font-weight: 700;">
      Incrementalizes
    </div>
    <div style="background: #E6F9F0; border: 2px solid #00A972; border-top: none; border-radius: 0 0 10px 10px; padding: 18px;">
      <table style="width: 100%; border-collapse: collapse;">
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">Simple aggregations</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;"><code style="font-size: 14pt;">SUM</code>, <code style="font-size: 14pt;">COUNT</code>, <code style="font-size: 14pt;">AVG</code>, <code style="font-size: 14pt;">MIN</code>, <code style="font-size: 14pt;">MAX</code></td></tr>
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">All JOIN types</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;">INNER, LEFT, RIGHT, FULL, CROSS</td></tr>
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">Filters</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;"><code style="font-size: 14pt;">WHERE</code>, <code style="font-size: 14pt;">HAVING</code>, <code style="font-size: 14pt;">QUALIFY</code></td></tr>
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">CASE expressions</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;">Deterministic conditional logic</td></tr>
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">Window functions</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;">With <code style="font-size: 14pt;">PARTITION BY</code></td></tr>
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">CTEs and subqueries</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #c8f0dc; font-size: 15pt !important; line-height: 1.6;">Standard <code style="font-size: 14pt;">WITH</code> clauses</td></tr>
        <tr><td style="padding: 8px 0; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">UNION ALL</strong></td><td style="padding: 8px 0; font-size: 15pt !important; line-height: 1.6;">Combining multiple sources</td></tr>
      </table>
    </div>
  </div>

  <!-- Does NOT incrementalize -->
  <div style="flex: 1; min-width: 400px;">
    <div style="background: #FF5F46; color: white; padding: 12px 18px; border-radius: 10px 10px 0 0; text-align: center; font-size: 15pt; font-weight: 700;">
<p style="font-size: 14pt; line-height: 1.6; color: #333;">Does NOT Incrementalize</p>
    </div>
    <div style="background: #FFF0EE; border: 2px solid #FF5F46; border-top: none; border-radius: 0 0 10px 10px; padding: 18px;">
      <table style="width: 100%; border-collapse: collapse;">
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #fdd; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">Non-deterministic functions</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #fdd; font-size: 15pt !important; line-height: 1.6;"><code style="font-size: 14pt;">UUID()</code>, <code style="font-size: 14pt;">RAND()</code>, <code style="font-size: 14pt;">RANDOM()</code></td></tr>
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #fdd; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">Non-Delta sources</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #fdd; font-size: 15pt !important; line-height: 1.6;">CSV, JSON, foreign catalogs</td></tr>
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #fdd; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">Row filters / column masks</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #fdd; font-size: 15pt !important; line-height: 1.6;">Security policies on source data</td></tr>
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #fdd; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">External UDFs</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #fdd; font-size: 15pt !important; line-height: 1.6;">Treated as non-deterministic</td></tr>
        <tr><td style="padding: 8px 0; border-bottom: 1px solid #fdd; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">Missing source properties</strong></td><td style="padding: 8px 0; border-bottom: 1px solid #fdd; font-size: 15pt !important; line-height: 1.6;">No row tracking or CDF enabled</td></tr>
        <tr><td style="padding: 8px 0; font-size: 15pt !important; line-height: 1.6;"><strong style="font-size: 14pt;">Volume / path-based sources</strong></td><td style="padding: 8px 0; font-size: 15pt !important; line-height: 1.6;">Files, not tables</td></tr>
      </table>
    </div>
  </div>

</div>

<!-- How to fix callout -->
<div style="margin-top: 20px; padding: 16px 20px; background: #FFF8E1; border: 2px solid #E5A100; border-radius: 10px;">
  <div style="font-size: 15pt; font-weight: 700; color: #1B3139; margin-bottom: 8px;">How to Fix Non-Incrementalizable Queries</div>
  <div style="font-size: 14pt; line-height: 1.7; color: #333;">
    <ul style="font-size: 14pt; margin: 0; padding-left: 20px;">
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Remove non-deterministic functions</strong>: if you need a UUID, generate it at query time or in a downstream view, not in the MV definition</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Enable source table properties</strong>: <code style="font-size: 14pt;">ALTER TABLE SET TBLPROPERTIES('delta.enableRowTracking'='true', 'delta.enableChangeDataFeed'='true', 'delta.enableDeletionVectors'='true')</code></li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Land non-Delta sources in Delta first</strong>: use a Streaming Table for file ingestion, then build the MV on the ST output</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Pre-aggregate before joining</strong>: restructure complex queries so each CTE step is incrementalizable</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Always re-run EXPLAIN after changes</strong>: verify the fix worked before deploying</li>
    </ul>
  </div>
</div>

</div>

<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">The Design-Time Mindset</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">In a traditional warehouse, you optimize queries after the fact with indexes and partitions. In a modern data warehouse, you optimize at <strong style="font-size: 14pt;">design time</strong> by writing SQL that Enzyme can incrementalize.</li>
          <li style="font-size: 14pt;">The practical impact is measurable: an incrementalizable MV over a terabyte table might refresh in seconds (processing only the 0.1% that changed), while a non-incrementalizable MV recomputes the entire terabyte on every refresh.</li>
          <li style="font-size: 14pt;">Think of incrementalizability as a <strong style="font-size: 14pt;">property of your SQL design</strong>, not a property of the data. Two queries that produce the same result can have very different refresh costs depending on how they are written.</li>
        </ul>
        <strong style="color: #1B5162;">Special Cases</strong>
        <ul style="font-size: 14pt; margin: 8px 0 12px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><code style="font-size: 14pt;">current_date()</code> and <code style="font-size: 14pt;">current_timestamp()</code> are special exceptions: they <strong style="font-size: 14pt;">are</strong> incrementalizable despite being technically non-deterministic, because the platform handles them as "deterministic within a refresh."</li>
          <li style="font-size: 14pt;"><code style="font-size: 14pt;">UNION</code> (with deduplication) may or may not incrementalize depending on the underlying complexity; prefer <code style="font-size: 14pt;">UNION ALL</code> when you know sources are disjoint.</li>
        </ul>
        <strong style="color: #1B5162;">Enzyme Refresh Techniques</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">When your query incrementalizes, Enzyme automatically chooses the cheapest refresh technique:
            <ul>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">ROW_BASED:</strong> Updates individual rows: best for small changesets</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">PARTITION_OVERWRITE:</strong> Rewrites only affected partitions: best for partitioned tables with localized changes</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">AGGREGATE_RECOMPUTE:</strong> Recomputes only affected aggregation groups</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">FULL_RECOMPUTE:</strong> Rewrites everything: the fallback when the above are not cheaper</li>
            </ul>
          </li>
          <li style="font-size: 14pt;">You can observe which technique was used by querying the event log for <code style="font-size: 14pt;">planning_information</code> events.</li>
          <li style="font-size: 14pt;">Full documentation on supported operations: (<a style="font-size: 14pt;" href="https://docs.databricks.com/aws/en/optimizations/incremental-refresh#support-for-materialized-view-incremental-refresh">AWS</a> | <a style="font-size: 14pt;" href="https://learn.microsoft.com/en-us/azure/databricks/optimizations/incremental-refresh#support-for-materialized-view-incremental-refresh">Azure</a> | <a style="font-size: 14pt;" href="https://docs.databricks.com/gcp/en/optimizations/incremental-refresh#support-for-materialized-view-incremental-refresh">GCP</a>)</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>

## F. Refresh Strategies as Business Decisions

REFRESH POLICY and SCHEDULE are more than technical options: they translate business SLA requirements into platform behavior. This section frames each policy and schedule as a <strong style="font-size: 14pt;">business decision</strong>.

### F1. REFRESH POLICY

<code style="font-size: 14pt;">REFRESH POLICY</code> controls the <em>how</em> of refresh: whether the Enzyme engine rebuilds only what changed (incremental) or rewrites the full result from scratch. Choosing the right policy lets you balance cost, freshness guarantees, and operational predictability.

<!-- ── Visual: tabs-color-cards (d2rp-) ── -->
<div class="d2rp-wrap">
<style>
.d2rp-wrap { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 100%; margin: 0 auto; padding: 24px; background: #FAFAFA; border: 1px solid #E0E0E0; border-radius: 12px; }
.d2rp-wrap input[type="radio"] { display: none; }
.d2rp-nav { display: flex; gap: 8px; flex-wrap: wrap; }
.d2rp-tab { flex: 1; min-width: 120px; text-align: center; padding: 10px 8px; font-size: 14pt; font-weight: 700; color: #1B3139; cursor: pointer; border: 2px solid #E0E0E0; border-radius: 6px; background: #FFF; transition: border-color 0.15s; user-select: none; position: relative; }
.d2rp-tab::before { content: ''; position: absolute; top: 0; left: 0; right: 0; height: 5px; border-radius: 4px 4px 0 0; }
.d2rp-tab[for="d2rp-r1"]::before { background: #2574B5; }
.d2rp-tab[for="d2rp-r2"]::before { background: #1B5162; }
.d2rp-tab[for="d2rp-r3"]::before { background: #E0382B; }
.d2rp-tab[for="d2rp-r4"]::before { background: #E5A100; }
#d2rp-r1:checked ~ .d2rp-nav .d2rp-tab[for="d2rp-r1"] { border-color: #2574B5; box-shadow: 0 0 0 1px #2574B5; }
#d2rp-r2:checked ~ .d2rp-nav .d2rp-tab[for="d2rp-r2"] { border-color: #1B5162; box-shadow: 0 0 0 1px #1B5162; }
#d2rp-r3:checked ~ .d2rp-nav .d2rp-tab[for="d2rp-r3"] { border-color: #E0382B; box-shadow: 0 0 0 1px #E0382B; }
#d2rp-r4:checked ~ .d2rp-nav .d2rp-tab[for="d2rp-r4"] { border-color: #E5A100; box-shadow: 0 0 0 1px #E5A100; }
.d2rp-accent { display: none; height: 5px; border-radius: 3px; margin: 16px 0 0 0; }
#d2rp-r1:checked ~ .d2rp-panels .d2rp-accent-1 { display: block; background: #2574B5; }
#d2rp-r2:checked ~ .d2rp-panels .d2rp-accent-2 { display: block; background: #1B5162; }
#d2rp-r3:checked ~ .d2rp-panels .d2rp-accent-3 { display: block; background: #E0382B; }
#d2rp-r4:checked ~ .d2rp-panels .d2rp-accent-4 { display: block; background: #E5A100; }
.d2rp-panel { display: none; padding: 20px 24px 24px; background: #F9F7F4; border: 1px solid #E8E3DC; border-radius: 0 0 8px 8px; }
#d2rp-r1:checked ~ .d2rp-panels .d2rp-panel:nth-of-type(2) { display: block; }
#d2rp-r2:checked ~ .d2rp-panels .d2rp-panel:nth-of-type(4) { display: block; }
#d2rp-r3:checked ~ .d2rp-panels .d2rp-panel:nth-of-type(6) { display: block; }
#d2rp-r4:checked ~ .d2rp-panels .d2rp-panel:nth-of-type(8) { display: block; }
.d2rp-panel h3 { margin: 0 0 12px; font-size: 16pt; color: #0b2026; font-weight: 800; }
.d2rp-panel ul { margin: 0 0 16px; padding-left: 20px; line-height: 1.7; color: #1B3139; font-size: 14pt; }
.d2rp-panel ul li { margin-bottom: 6px; }
.d2rp-panel code { background: #E8E3DC; padding: 2px 6px; border-radius: 4px; font-size: 14pt; }
.d2rp-callout { background: #E8F0E4; border-left: 4px solid #1B5162; padding: 14px 18px; border-radius: 0 6px 6px 0; margin-top: 8px; font-size: 14pt; line-height: 1.6; }
.d2rp-callout strong { color: #0b2026; }
.d2rp-callout code { background: transparent; padding: 0; font-size: 14pt; }
.d2rp-code { background: #272822; border-radius: 8px; padding: 14px 18px; margin-top: 10px; font-family: monospace; font-size: 14pt; line-height: 1.8; color: #f8f8f2; }
.d2rp-code .kw { color: #66d9ef; font-weight: 600; }
.d2rp-code .pol { color: #a6e22e; font-weight: 600; }
/* ── EXPLAIN × Policy matrix ── */
/* ── EXPLAIN × Policy two-pane ── */
.d2rp-ex { display: flex; gap: 0; margin-top: 16px; border-radius: 10px; overflow: hidden; font-size: 14pt; line-height: 1.5; }
.d2rp-ex-yes { flex: 1; background: #E6F9F0; border-left: 4px solid #1B5162; padding: 12px 16px; }
.d2rp-ex-no  { flex: 1; background: #FFF0EE; border-left: 4px solid #FF5F46; padding: 12px 16px; }
.d2rp-ex-label { font-size: 14pt; font-weight: 700; margin-bottom: 4px; }
.d2rp-ex-yes .d2rp-ex-label { color: #00A972; }
.d2rp-ex-no  .d2rp-ex-label { color: #FF5F46; }
.d2rp-ex-text { color: #1B3139; }
</style>
<input type="radio" name="d2rp-grp" id="d2rp-r1" checked>
<input type="radio" name="d2rp-grp" id="d2rp-r2">
<input type="radio" name="d2rp-grp" id="d2rp-r3">
<input type="radio" name="d2rp-grp" id="d2rp-r4">
<div class="d2rp-nav">
  <label class="d2rp-tab" for="d2rp-r1">AUTO (default)</label>
  <label class="d2rp-tab" for="d2rp-r2">INCREMENTAL</label>
  <label class="d2rp-tab" for="d2rp-r3">INCREMENTAL STRICT</label>
  <label class="d2rp-tab" for="d2rp-r4">FULL</label>
</div>
<div class="d2rp-panels">
  <div class="d2rp-accent d2rp-accent-1"></div>
  <div class="d2rp-panel">
    <h3>REFRESH POLICY AUTO (Default)</h3>
    <ul>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">What it does:</strong> The Enzyme <strong>cost model decides</strong> incremental vs. full on each individual refresh</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">When to use it:</strong> Best <strong>starting point</strong>: no configuration needed; works correctly out of the box</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">How it decides:</strong> If a <strong>small fraction</strong> of rows changed, the model <strong><em>picks incremental</em></strong>; if <strong>schema changed</strong> or the changeset is large, it <strong><em>picks full</em></strong></li>
    </ul>
    <div class="d2rp-code"><span class="kw">CREATE MATERIALIZED VIEW</span> gold_revenue<br/><span class="pol">REFRESH POLICY AUTO</span><br/><span class="kw">AS SELECT</span> product, <span class="kw">SUM</span>(quantity) <span class="kw">AS</span> total_qty<br/><span class="kw">FROM</span> sales_transactions <span class="kw">GROUP BY</span> product;</div>
    <div class="d2rp-ex">
      <div class="d2rp-ex-yes"><div class="d2rp-ex-label"> Incrementalizable</div><div class="d2rp-ex-text"><strong style="color:#00795b;">Cost model decides</strong> each refresh. Incremental when efficient, full when not</div></div>
      <div class="d2rp-ex-no"><div class="d2rp-ex-label"> Not Incrementalizable</div><div class="d2rp-ex-text"><strong style="color:#E5A100;">Full recompute</strong> every time</div></div>
    </div>
  </div>
  <div class="d2rp-accent d2rp-accent-2"></div>
  <div class="d2rp-panel">
    <h3>REFRESH POLICY INCREMENTAL</h3>
    <ul>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">What it does:</strong> <strong>Prefers incremental</strong> refresh at runtime; <strong>falls back to full</strong> only when a specific refresh cannot be done incrementally (e.g., after a schema change)</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">When to use it:</strong> When you want incremental as the default but can <strong>tolerate occasional full recomputes</strong> at runtime</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Requires incrementalizable query:</strong> <code style="font-size: 14pt;">CREATE</code> <strong>fails immediately</strong> if the query itself is not incrementalizable: run <code style="font-size: 14pt;">EXPLAIN</code> first</li>
    </ul>
    <div class="d2rp-code"><span class="kw">CREATE MATERIALIZED VIEW</span> gold_revenue<br/><span class="pol">REFRESH POLICY INCREMENTAL</span><br/><span class="kw">AS SELECT</span> product, <span class="kw">SUM</span>(quantity) <span class="kw">AS</span> total_qty<br/><span class="kw">FROM</span> sales_transactions <span class="kw">GROUP BY</span> product;</div>
    <div class="d2rp-ex">
      <div class="d2rp-ex-yes"><div class="d2rp-ex-label"> Incrementalizable</div><div class="d2rp-ex-text"><strong style="color:#00795b;">Incremental preferred</strong>, with fallback to full if needed</div></div>
      <div class="d2rp-ex-no"><div class="d2rp-ex-label"> Not Incrementalizable</div><div class="d2rp-ex-text"><strong style="color:#E0382B;">CREATE fails immediately</strong></div></div>
    </div>
  </div>
  <div class="d2rp-accent d2rp-accent-3"></div>
  <div class="d2rp-panel">
    <h3>REFRESH POLICY INCREMENTAL STRICT</h3>
    <ul>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">What it does:</strong> Requires <strong>incremental on every run</strong>: <strong>fails</strong> rather than falling back to full</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">When to use it:</strong> Cost-sensitive pipelines where a <strong>surprise full recompute</strong> would be unacceptable</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">No cheat days:</strong> If incremental is not possible for any reason, the refresh <strong>fails with an error</strong> instead of silently doing a full recompute</li>
    </ul>
    <div class="d2rp-code"><span class="kw">CREATE MATERIALIZED VIEW</span> gold_revenue<br/><span class="pol">REFRESH POLICY INCREMENTAL STRICT</span><br/><span class="kw">AS SELECT</span> product, <span class="kw">SUM</span>(quantity) <span class="kw">AS</span> total_qty<br/><span class="kw">FROM</span> sales_transactions <span class="kw">GROUP BY</span> product;</div>
    <div class="d2rp-ex">
      <div class="d2rp-ex-yes"><div class="d2rp-ex-label"> Incrementalizable</div><div class="d2rp-ex-text"><strong style="color:#00795b;">Incremental only</strong>: fails rather than falling back to full</div></div>
      <div class="d2rp-ex-no"><div class="d2rp-ex-label"> Not Incrementalizable</div><div class="d2rp-ex-text"><strong style="color:#E0382B;">CREATE fails immediately</strong></div></div>
    </div>
  </div>
  <div class="d2rp-accent d2rp-accent-4"></div>
  <div class="d2rp-panel">
    <h3>REFRESH POLICY FULL</h3>
    <ul>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">What it does:</strong> Always rewrites the <strong>entire result from scratch</strong>, regardless of how much source data changed</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">When to use it:</strong> Queries with <strong>non-deterministic functions</strong>, or when you need a guaranteed <strong>clean slate</strong> (e.g., after source corrections)</li>
      <li style="font-size: 14pt;"><strong style="font-size: 14pt;">The escape hatch:</strong> Works on <strong>any query</strong>: even ones that are not incrementalizable. No EXPLAIN check required</li>
    </ul>
    <div class="d2rp-code"><span class="kw">CREATE MATERIALIZED VIEW</span> gold_revenue<br/><span class="pol">REFRESH POLICY FULL</span><br/><span class="kw">AS SELECT</span> product, <span class="kw">SUM</span>(quantity) <span class="kw">AS</span> total_qty<br/><span class="kw">FROM</span> sales_transactions <span class="kw">GROUP BY</span> product;</div>
    <div class="d2rp-ex">
      <div class="d2rp-ex-yes"><div class="d2rp-ex-label"> Incrementalizable</div><div class="d2rp-ex-text"><strong style="color:#E5A100;">Full recompute</strong>: ignores incremental capability</div></div>
      <div class="d2rp-ex-no"><div class="d2rp-ex-label"> Not Incrementalizable</div><div class="d2rp-ex-text"><strong style="color:#E5A100;">Full recompute</strong>: the only option that works</div></div>
    </div>
  </div>
</div>

</div>

<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">The Four Refresh Policies</strong>
        <ul style="font-size: 14pt; margin: 8px 0 14px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">AUTO (default):</strong> The Enzyme cost model evaluates the changeset on every refresh and picks the cheapest strategy: incremental if the delta is small, full if it is large or if a schema change was detected. You get optimal cost without any manual tuning.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">INCREMENTAL:</strong> Tells the platform to prefer incremental but permits a <strong>full recompute as a fallback</strong> when a specific refresh cannot be done incrementally (e.g., after a schema change). The <code style="font-size: 14pt;">CREATE</code> statement itself <strong>fails immediately</strong> if the query is structurally non-incrementalizable, so you get an early error rather than a silent full refresh at runtime.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">INCREMENTAL STRICT:</strong> Requires incremental on <strong>every single refresh</strong>. If anything prevents incremental (schema change, source table property missing), the refresh <strong>fails with an explicit error</strong> instead of silently falling back. Use this when a surprise full recompute would breach your SLA or budget.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">FULL:</strong> Always rewrites the entire result from scratch. The simplest guarantee, and the <strong>only option</strong> for queries that use non-deterministic functions like <code style="font-size: 14pt;">UUID()</code> or <code style="font-size: 14pt;">RANDOM()</code>. Also useful after correcting source data when you want a guaranteed clean slate.</li>
        </ul>
        <strong style="color: #1B5162;">Connecting EXPLAIN to REFRESH POLICY</strong>
        <ul style="font-size: 14pt; margin: 8px 0 14px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Remember the critical distinction from D1: when EXPLAIN says <em>"The Materialized View can be incrementally refreshed"</em>, that means the query is <strong style="font-size: 14pt;">incrementalizable</strong> (structurally capable), not that it <strong style="font-size: 14pt;">will be incrementalized</strong> at every refresh. REFRESH POLICY is how you control what happens next:
            <ul>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Incrementalizable + AUTO:</strong> Cost model decides each refresh: incremental when efficient, full when not</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Incrementalizable + INCREMENTAL:</strong> Incremental preferred, with fallback to full if needed</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Incrementalizable + INCREMENTAL STRICT:</strong> Incremental only: fails rather than falling back to full</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Not incrementalizable + INCREMENTAL/STRICT:</strong> <code style="font-size: 14pt;">CREATE</code> fails immediately: you cannot force an incremental policy on a query that does not support it</li>
            </ul>
          </li>
          <li style="font-size: 14pt;">If you attempt <code style="font-size: 14pt;">INCREMENTAL</code> on a non-incrementalizable query, you will see the error class <code style="font-size: 14pt;">MATERIALIZED_VIEW_NOT_INCREMENTALIZABLE</code> with sub-reasons like <code style="font-size: 14pt;">OPERATOR_NOT_SUPPORTED</code> or <code style="font-size: 14pt;">EXPRESSION_NOT_DETERMINISTIC</code>. This is why D1 recommended running <code style="font-size: 14pt;">EXPLAIN</code> before choosing your policy: it catches these issues at development time, not deployment time.</li>
          <li style="font-size: 14pt;">Think of it as two gates: EXPLAIN is the <strong style="font-size: 14pt;">first gate</strong> (can the query be incrementalized at all?), and REFRESH POLICY is the <strong style="font-size: 14pt;">second gate</strong> (how strictly do you want to enforce it at runtime?).</li>
        </ul>
        <strong style="color: #1B5162;">How the Cost Model Decides (AUTO)</strong>
        <ul style="font-size: 14pt; margin: 8px 0 14px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Under <code style="font-size: 14pt;">AUTO</code>, the Enzyme engine computes an estimated cost for both strategies before each refresh and picks the lower one. Key inputs include:
            <ul>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Changeset size:</strong> If only a small fraction of source rows changed, incremental reads only those rows; if nearly all rows changed, full is cheaper because it avoids tracking overhead</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Schema changes:</strong> Any structural change on a source table (column added, type altered) typically forces a full recompute because the incremental plan becomes invalid</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Source table properties:</strong> Without <code style="font-size: 14pt;">delta.enableChangeDataFeed</code> and <code style="font-size: 14pt;">delta.enableRowTracking</code>, the engine has no change feed to read; it falls back to full automatically</li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Operation type:</strong> Some aggregations (e.g., certain window functions) have higher incremental overhead than others, shifting the cost model toward full</li>
            </ul>
          </li>
          <li style="font-size: 14pt;">You can observe what the cost model decided for each refresh by querying the event log for <code style="font-size: 14pt;">planning_information</code> events: the message will show the Enzyme technique (e.g., <code style="font-size: 14pt;">ROW_BASED</code>, <code style="font-size: 14pt;">FULL_RECOMPUTE</code>).</li>
        </ul>
        <strong style="color: #1B5162;">In Practice</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Start with AUTO:</strong> For any new MV, <code style="font-size: 14pt;">REFRESH POLICY AUTO</code> is the correct default. It requires no configuration, picks the cheapest strategy on each refresh, and handles edge cases gracefully.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Move to INCREMENTAL when you want a guarantee:</strong> Once you have verified via <code style="font-size: 14pt;">EXPLAIN</code> and observed in the event log that your MV is consistently doing incremental refreshes, switch to <code style="font-size: 14pt;">INCREMENTAL</code> to make that intent explicit and get an early error if something changes.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Use INCREMENTAL STRICT for cost-critical pipelines:</strong> If your budget depends on never doing a full recompute; for example, an MV over a petabyte-scale table where a full refresh would cost thousands of DBUs. <code style="font-size: 14pt;">INCREMENTAL STRICT</code> makes the failure loud and explicit rather than silently expensive.</li>
          <li style="font-size: 14pt;"><strong style="font-size: 14pt;">FULL as the escape hatch:</strong> Use <code style="font-size: 14pt;">FULL</code> when the query cannot be incrementalized (non-deterministic functions, foreign catalogs) or when correctness after source corrections is more important than refresh cost.</li>
          <li style="font-size: 14pt;">The official documentation confirms: both <code style="font-size: 14pt;">INCREMENTAL</code> and <code style="font-size: 14pt;">INCREMENTAL STRICT</code> fail at <code style="font-size: 14pt;">CREATE</code> time if the query is not incrementalizable. The difference is only at <strong>refresh time</strong>: <code style="font-size: 14pt;">INCREMENTAL</code> falls back to full, <code style="font-size: 14pt;">STRICT</code> fails entirely. (docs: <a style="font-size: 14pt;" href="https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-materialized-view-refresh-policy">AWS</a> | <a style="font-size: 14pt;" href="https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-create-materialized-view-refresh-policy">Azure</a> | <a style="font-size: 14pt;" href="https://docs.databricks.com/gcp/en/sql/language-manual/sql-ref-syntax-ddl-create-materialized-view-refresh-policy">GCP</a>)</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>

### F2. SCHEDULE and TRIGGER

<code style="font-size: 14pt;">SCHEDULE</code> controls the <em>when</em> of refresh: completely independently of which <code style="font-size: 14pt;">REFRESH POLICY</code> you chose. You can pair any policy with any schedule, and you can change the schedule after creation without touching the MV definition.

<!-- ── Visual: three-pane cards (d3sc-) ── -->
<div class="d3sc-wrap">
<style>
.d3sc-wrap { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 100%; margin: 0 auto; }
.d3sc-cards { display: flex; gap: 16px; flex-wrap: wrap; }
.d3sc-card { flex: 1; min-width: 240px; background: #F9F7F4; border: 1px solid #E8E3DC; border-radius: 10px; overflow: hidden; }
.d3sc-card-hdr { padding: 14px 18px 10px; color: #fff; font-size: 15pt; font-weight: 700; }
.d3sc-card-body { padding: 14px 18px 18px; font-size: 14pt; line-height: 1.7; color: #1B3139; }
.d3sc-card-body ul { margin: 8px 0 12px 0; padding-left: 20px; }
.d3sc-card-body li { margin-bottom: 6px; }
.d3sc-card-body code { background: #E8E3DC; padding: 2px 5px; border-radius: 3px; font-size: 14pt; }
.d3sc-syntax { background: #272822; color: #f8f8f2; border-radius: 6px; padding: 10px 14px; font-family: monospace; font-size: 14pt; line-height: 1.6; margin-top: 10px; white-space: pre-wrap; }
.d3sc-syntax .kw { color: #66d9ef; font-weight: 600; }
.d3sc-syntax .str { color: #e6db74; }
</style>
<div class="d3sc-cards">
  <div class="d3sc-card">
    <div class="d3sc-card-hdr" style="background: #2574B5">SCHEDULE EVERY</div>
    <div class="d3sc-card-body">
      <ul>
        <li style="font-size: 14pt;">Refresh at a <strong>fixed interval</strong>: no cron syntax needed</li>
        <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Best for:</strong> Simple pipelines with regular cadence</li>
        <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Units:</strong> <code style="font-size: 14pt;">SECONDS</code>, <code style="font-size: 14pt;">MINUTES</code>, <code style="font-size: 14pt;">HOURS</code>, <code style="font-size: 14pt;">DAYS</code>, <code style="font-size: 14pt;">WEEKS</code></li>
      </ul>
      <div class="d3sc-syntax"><span class="kw">CREATE OR REPLACE MATERIALIZED VIEW</span> mv_hourly<br/>  <span class="kw">SCHEDULE EVERY</span> 1 HOUR<br/><span class="kw">AS SELECT</span> ...</div>
    </div>
  </div>
  <div class="d3sc-card">
    <div class="d3sc-card-hdr" style="background: #1B5162">SCHEDULE CRON</div>
    <div class="d3sc-card-body">
      <ul>
        <li style="font-size: 14pt;">Precise cron expression with <strong>required timezone</strong></li>
        <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Best for:</strong> Weekdays at 6 AM, end-of-month, business hours</li>
        <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Gotcha:</strong> Omitting <code style="font-size: 14pt;">AT TIME ZONE</code> throws an error</li>
      </ul>
      <div class="d3sc-syntax"><span class="kw">CREATE OR REPLACE MATERIALIZED VIEW</span> mv_weekday<br/>  <span class="kw">SCHEDULE CRON</span> <span class="str">'0 0 6 ? * MON-FRI'</span><br/>  <span class="kw">AT TIME ZONE</span> <span class="str">'America/New_York'</span><br/><span class="kw">AS SELECT</span> ...</div>
    </div>
  </div>
  <div class="d3sc-card">
    <div class="d3sc-card-hdr" style="background: #E5A100">TRIGGER ON UPDATE</div>
    <div class="d3sc-card-body">
      <ul>
        <li style="font-size: 14pt;"><strong>Event-driven</strong>: refreshes when upstream sources change</li>
        <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Best for:</strong> Near-real-time sync without polling</li>
        <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Throttle:</strong> <code style="font-size: 14pt;">AT MOST EVERY</code> prevents over-refreshing</li>
      </ul>
      <div class="d3sc-syntax"><span class="kw">CREATE OR REPLACE MATERIALIZED VIEW</span> mv_trigger<br/>  <span class="kw">TRIGGER ON UPDATE</span><br/>  <span class="kw">AT MOST EVERY</span> 5 MINUTES<br/><span class="kw">AS SELECT</span> ...<br/><span class="kw">FROM</span> base_table</div>
    </div>
  </div>
</div>
</div>

<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none;">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span style="font-size: 20px;">&#x25B6;</span>
        <strong style="color: #1B5162; font-size: 14pt;">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="font-size: 20px; visibility: hidden;">&#x25B6;</span>
      <div>
        <strong style="color: #1B5162;">SCHEDULE EVERY: Simple Intervals</strong>
        <ul style="font-size: 14pt; margin: 8px 0 14px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">You specify a number and a time unit (<code style="font-size: 14pt;">SECONDS</code>, <code style="font-size: 14pt;">MINUTES</code>, <code style="font-size: 14pt;">HOURS</code>, <code style="font-size: 14pt;">DAYS</code>, <code style="font-size: 14pt;">WEEKS</code>) and Databricks handles the rest. No cron syntax to learn.</li>
          <li style="font-size: 14pt;">The interval is measured from the <strong>end of the previous refresh</strong>, not from a fixed wall-clock anchor; so <code style="font-size: 14pt;">EVERY 1 HOUR</code> means "1 hour after the last refresh completed," not "on the hour."</li>
          <li style="font-size: 14pt;">Best for pipelines where you want a consistent cadence and do not need to control the exact time of day.</li>
        </ul>
        <strong style="color: #1B5162;">SCHEDULE CRON: Precise Timing</strong>
        <ul style="font-size: 14pt; margin: 8px 0 14px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Uses standard Quartz cron syntax for precise scheduling: weekday mornings, end-of-month, intra-day windows tied to a specific region.</li>
          <li style="font-size: 14pt;">You <strong>must</strong> include <code style="font-size: 14pt;">AT TIME ZONE</code>. If you omit it and no session timezone is set, the statement throws an error. Always use a canonical timezone like <code style="font-size: 14pt;">'UTC'</code> or <code style="font-size: 14pt;">'America/New_York'</code>: relying on an implicit session timezone leads to surprises when users in different regions share the same MV.</li>
          <li style="font-size: 14pt;">Think of SCHEDULE EVERY as an <strong style="font-size: 14pt;">egg timer</strong> (relative intervals) and SCHEDULE CRON as an <strong style="font-size: 14pt;">alarm clock</strong> (fixed wall-clock times). Choose the one that matches your SLA.</li>
        </ul>
        <strong style="color: #1B5162;">TRIGGER ON UPDATE. Event-Driven</strong>
        <ul style="font-size: 14pt; margin: 8px 0 14px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">Instead of running on a clock, TRIGGER fires when an <strong>upstream source table is modified</strong>. Think of it as a <strong style="font-size: 14pt;">doorbell</strong>: the refresh rings when data arrives, rather than checking on a timer.</li>
          <li style="font-size: 14pt;">This is <strong>not continuous streaming</strong>: it launches a discrete refresh when a change is detected. For true sub-minute latency, use a Streaming Table instead.</li>
          <li style="font-size: 14pt;">The optional <code style="font-size: 14pt;">AT MOST EVERY</code> throttle prevents a high-frequency source from triggering hundreds of refreshes per hour. Minimum interval is <strong style="font-size: 14pt;">1 minute</strong>.</li>
        </ul>
        <strong style="color: #1B5162;">Combining Policy and Schedule</strong>
        <ul style="font-size: 14pt; margin: 8px 0 14px 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;"><code style="font-size: 14pt;">REFRESH POLICY</code> (the <em>how</em>) and <code style="font-size: 14pt;">SCHEDULE</code> (the <em>when</em>) are completely orthogonal: you can pair any combination:
            <ul>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">AUTO + hourly:</strong> <code style="font-size: 14pt;">REFRESH POLICY AUTO SCHEDULE EVERY 1 HOUR</code></li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">INCREMENTAL + weekday mornings:</strong> <code style="font-size: 14pt;">REFRESH POLICY INCREMENTAL SCHEDULE CRON '0 0 6 ? * MON-FRI' AT TIME ZONE 'America/New_York'</code></li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">STRICT + event-driven:</strong> <code style="font-size: 14pt;">REFRESH POLICY INCREMENTAL STRICT TRIGGER ON UPDATE AT MOST EVERY 5 MINUTES</code></li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">FULL + nightly:</strong> <code style="font-size: 14pt;">REFRESH POLICY FULL SCHEDULE CRON '0 0 2 * * ?' AT TIME ZONE 'UTC'</code></li>
            </ul>
          </li>
          <li style="font-size: 14pt;">When a schedule is active, Databricks automatically creates a managed <strong style="font-size: 14pt;">Lakeflow Job</strong>: visible in Jobs &amp; Pipelines. Your warehouse does <strong>not</strong> need to be running for scheduled refreshes to execute.</li>
        </ul>
        <strong style="color: #1B5162;">Managing Schedules After Creation</strong>
        <ul style="font-size: 14pt; margin: 8px 0 0 0; color: #333; line-height: 1.7">
          <li style="font-size: 14pt;">You can add, modify, or drop a schedule after the MV is created: no need to recreate:
            <ul>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Add:</strong> <code style="font-size: 14pt;">ALTER MATERIALIZED VIEW mv ADD SCHEDULE CRON '...' AT TIME ZONE '...'</code></li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Change:</strong> <code style="font-size: 14pt;">ALTER MATERIALIZED VIEW mv ALTER SCHEDULE CRON '...' AT TIME ZONE '...'</code></li>
              <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Remove:</strong> <code style="font-size: 14pt;">ALTER MATERIALIZED VIEW mv DROP SCHEDULE</code></li>
            </ul>
          </li>
          <li style="font-size: 14pt;">Manual refresh always works, even when a schedule is active: <code style="font-size: 14pt;">REFRESH MATERIALIZED VIEW my_mv</code> (blocking) or <code style="font-size: 14pt;">REFRESH MATERIALIZED VIEW my_mv ASYNC</code> (non-blocking).</li>
          <li style="font-size: 14pt;">If a refresh is still running when the next schedule fires, the overlapping execution is <strong>skipped</strong>: no duplicate refreshes, no queuing.</li>
        </ul>
      </div>
    </div>
  </div>
</details>

<style>
details summary span:first-child {
  transition: transform 0.2s ease;
  display: inline-block;
}
details[open] summary span:first-child {
  transform: rotate(90deg);
}
</style>

## Conclusion

Streaming Tables handle ingestion; Materialized Views handle transformations. The tools below make them production-ready.

<p style="font-size: 14pt; line-height: 1.6; color: #333;">Here is what to remember:</p>
<ul style="font-size: 14pt; line-height: 1.7">
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">Streaming Tables ingest incrementally:</strong> A Streaming Table wraps Auto Loader via <code style="font-size: 14pt;">read_files()</code>, providing exactly-once processing, automatic schema evolution, and a rescued data column, all on serverless compute separate from your warehouse.</li>
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">Auto Loader replaces COPY INTO:</strong> COPY INTO is listing-based and degrades at scale. Auto Loader uses checkpoint-based tracking with file notification mode, handling millions of files; Streaming Tables make it accessible with a single SQL statement.</li>
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">Materialized Views pre-compute and refresh incrementally:</strong> An MV physically stores query results as a table and uses the Enzyme engine to process only what changed: same answer as a full recompute, far less compute.</li>
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">MVs replace static Views for analytics:</strong> Views are pay-per-query (full scan every time); MVs are pay-per-refresh (compute once, read many). For BI dashboards with concurrent users, the cost savings add up quickly.</li>
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">EXPLAIN validates before deployment:</strong> <code style="font-size: 14pt;">EXPLAIN CREATE MATERIALIZED VIEW</code> is a dry-run check that confirms structural eligibility for incrementalization without creating the MV. Use it in combination with <code style="font-size: 14pt;">REFRESH POLICY</code> for full control.</li>
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">REFRESH POLICY controls the how:</strong> Four policies: AUTO lets the Enzyme cost model decide each refresh; INCREMENTAL prefers incremental with a full fallback; INCREMENTAL STRICT enforces incremental with no fallback (fails rather than surprises you); FULL always rewrites from scratch.</li>
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">SCHEDULE controls the when, independently:</strong> SCHEDULE EVERY for simple intervals, SCHEDULE CRON for precise time-aligned runs, and TRIGGER ON UPDATE for event-driven refresh when upstream sources change. Policy and schedule are orthogonal: combine any pair to match your SLA.</li>
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">Choose ST for ingestion, MV for transformations:</strong> Streaming Tables handle append-only ingestion and CDC. Materialized Views handle aggregations, joins, and business logic. Together they power the Bronze&ndash;Silver&ndash;Gold medallion pattern.</li>
<li style="font-size: 14pt;"><strong style="font-size: 14pt;">Serverless compute powers everything under the hood:</strong> Every MV and ST runs on automatically provisioned serverless infrastructure with incremental refresh, accessible to SQL analysts via DDL statements.</li>
</ul>

<p style="font-size: 14pt; line-height: 1.6; color: #333;">In the next demo (Demo 2.2), you will get hands-on with these building blocks, creating your first Streaming Tables and Materialized Views against the course dataset.</p>

<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <strong style="color: #37474f; font-size: 14pt;">Additional Context</strong>
  <ul style="font-size: 14pt; margin: 8px 0 0 20px; color: #333;">
    <li style="font-size: 14pt;">Use Streaming Tables in the Data Warehouse (<a style="font-size: 14pt;" href="https://docs.databricks.com/aws/en/ldp/dbsql/streaming">AWS</a> | <a style="font-size: 14pt;" href="https://learn.microsoft.com/en-us/azure/databricks/ldp/dbsql/streaming">Azure</a> | <a style="font-size: 14pt;" href="https://docs.databricks.com/gcp/en/ldp/dbsql/streaming">GCP</a>): creating, scheduling, and refreshing Streaming Tables with Auto Loader on serverless compute</li>
    <li style="font-size: 14pt;">Use Materialized Views in the Data Warehouse (<a style="font-size: 14pt;" href="https://docs.databricks.com/aws/en/ldp/dbsql/materialized">AWS</a> | <a style="font-size: 14pt;" href="https://learn.microsoft.com/en-us/azure/databricks/ldp/dbsql/materialized">Azure</a> | <a style="font-size: 14pt;" href="https://docs.databricks.com/gcp/en/ldp/dbsql/materialized">GCP</a>): creating, refreshing, and monitoring Materialized Views with Enzyme incremental refresh</li>
    <li style="font-size: 14pt;">Incremental Refresh for Materialized Views (<a style="font-size: 14pt;" href="https://docs.databricks.com/aws/en/optimizations/incremental-refresh#support-for-materialized-view-incremental-refresh">AWS</a> | <a style="font-size: 14pt;" href="https://learn.microsoft.com/en-us/azure/databricks/optimizations/incremental-refresh#support-for-materialized-view-incremental-refresh">Azure</a> | <a style="font-size: 14pt;" href="https://docs.databricks.com/gcp/en/optimizations/incremental-refresh#support-for-materialized-view-incremental-refresh">GCP</a>): supported operations, refresh techniques, and source table requirements for Enzyme</li>
    <li style="font-size: 14pt;">EXPLAIN CREATE MATERIALIZED VIEW (<a style="font-size: 14pt;" href="https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-explain-materialized-view">AWS</a> | <a style="font-size: 14pt;" href="https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-qry-explain-materialized-view">Azure</a> | <a style="font-size: 14pt;" href="https://docs.databricks.com/gcp/en/sql/language-manual/sql-ref-syntax-qry-explain-materialized-view">GCP</a>): the pre-deployment dry-run command for predicting incrementalization eligibility</li>
    <li style="font-size: 14pt;">REFRESH POLICY Reference (<a style="font-size: 14pt;" href="https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-materialized-view-refresh-policy">AWS</a> | <a style="font-size: 14pt;" href="https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-create-materialized-view-refresh-policy">Azure</a> | <a style="font-size: 14pt;" href="https://docs.databricks.com/gcp/en/sql/language-manual/sql-ref-syntax-ddl-create-materialized-view-refresh-policy">GCP</a>): official documentation for AUTO, INCREMENTAL, INCREMENTAL STRICT, and FULL policies</li>
    <li style="font-size: 14pt;">Spark Declarative Pipelines Concepts (<a style="font-size: 14pt;" href="https://docs.databricks.com/aws/en/ldp/concepts">AWS</a> | <a style="font-size: 14pt;" href="https://learn.microsoft.com/en-us/azure/databricks/ldp/concepts">Azure</a> | <a style="font-size: 14pt;" href="https://docs.databricks.com/gcp/en/ldp/concepts">GCP</a>): the serverless framework architecture behind Materialized Views and Streaming Tables</li>
  </ul>
</div>

&copy; 2026 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/>
<a href="https://databricks.com/privacy-policy">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use">Terms of Use</a> | <a href="https://help.databricks.com/">Support</a>